In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate momentum, efficiency, relative performance, and factor exposure for a single asset.
#Original Risk Analysis blocks included here: 14-21.


In [ ]:
# Block 2: import libraries and initialize analytics services
import logging
import warnings
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()
SINGLE_ASSET_DIRECTORY = PROJECT_ROOT / "Research" / "Single Asset"
if str(SINGLE_ASSET_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SINGLE_ASSET_DIRECTORY))
from _params import get_single_asset_params

from Quantapp.visualization import (
    Plotter,
    )
from Quantapp.visualization.core import (
    configure_plotly_notebook_renderers,
    )
from Quantapp.visualization.views.single_asset_profile.pricing.momentum_efficiency import (
    plot_benchmark_zscore_detail,
    plot_candlestick_drawdown_recovery_view,
    plot_momentum_zscore_comparison,
    plot_momentum_window_diagnostics_grid_view,
    plot_rolling_correlation_view,
    plot_seasonality_stack_view,
    plot_sharpe_sortino_comparison,
    plot_sharpe_surface_view,
    plot_sharpe_zscore_heatmap_view,
    )
from Quantapp.analytics import compute
from Quantapp.analytics import (
    Metric,
    SeriesTransforms,
    )
from Quantapp.data import GICSDataClient, build_gics_peer_frames, get_market_history

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
metric = Metric()
series_transforms = SeriesTransforms()

def risk_adjusted_returns(data, windows, ratio_type='sharpe', risk_free_rate=0.0, annualization_factor=252):
    if isinstance(windows, (str, bytes)):
        raise ValueError("windows must be an integer or an iterable of integers")
    try:
        window_list = [int(window) for window in windows]
    except TypeError:
        window_list = [int(windows)]
    if not window_list or any(window <= 0 for window in window_list):
        raise ValueError("windows must contain positive integers")

    price_frame = data.to_frame(name=data.name or "price") if isinstance(data, pd.Series) else data
    if not isinstance(price_frame, pd.DataFrame):
        raise TypeError("data must be a pandas Series or DataFrame")

    returns = price_frame.pct_change()
    if isinstance(risk_free_rate, pd.Series):
        periodic_rate = risk_free_rate.astype(float).sort_index().reindex(returns.index).ffill()
    elif np.isscalar(risk_free_rate):
        periodic_rate = pd.Series((1.0 + float(risk_free_rate)) ** (1.0 / annualization_factor) - 1.0, index=returns.index)
    else:
        raise TypeError("risk_free_rate must be a scalar annual rate or a pandas Series")

    excess_returns = returns.sub(periodic_rate, axis=0)
    single_window = len(window_list) == 1
    single_series = price_frame.shape[1] == 1
    output = []
    for column in returns.columns:
        excess = excess_returns[column]
        for window in window_list:
            mean_excess = excess.rolling(window).mean()
            if ratio_type == 'sharpe':
                volatility = excess.rolling(window).std()
                ratio = np.sqrt(annualization_factor) * mean_excess / volatility
                ratio = ratio.where(volatility > 0)
            elif ratio_type == 'sortino':
                downside = excess.where(excess < 0, 0.0)
                downside_deviation = downside.rolling(window).apply(lambda values: np.sqrt((values**2).mean()), raw=True)
                ratio = np.sqrt(annualization_factor) * mean_excess / downside_deviation
            else:
                raise ValueError("Invalid ratio_type. Use 'sharpe' or 'sortino'.")

            ratio = ratio.replace([np.inf, -np.inf], np.nan)
            ratio.name = f"{ratio_type}_ratio_{window}" if single_window and single_series else f"{column}_{ratio_type}_{window}"
            output.append(ratio)
    return pd.concat(output, axis=1)


In [ ]:
# Block 3: initialize plotting helpers and Momentum & Efficiency display theme

import plotly.graph_objects as go

qp = Plotter()

# This is intentionally notebook-local: importing Quantapp.visualization does not apply this theme.
configure_plotly_notebook_renderers()

CURRENT_VALUE_REFERENCE_META_KEY = "quantapp_current_value_reference"
CURRENT_VALUE_REFERENCE_LINE_STYLE = dict(color="rgba(250, 204, 21, 0.82)", width=2, dash="dash")

def _is_current_value_reference_trace(trace):
    meta = getattr(trace, "meta", None)
    if isinstance(meta, dict) and meta.get(CURRENT_VALUE_REFERENCE_META_KEY):
        return True
    return str(getattr(trace, "name", "")).endswith(" Current Value")

def _current_value_reference_visibility(source_visible):
    return False if source_visible is False else "legendonly"

def _numeric_trace_y(trace):
    y_values = getattr(trace, "y", None)
    if y_values is None:
        return None, None
    numeric_y = pd.to_numeric(pd.Series(list(y_values)), errors="coerce")
    numeric_array = numeric_y.to_numpy(dtype=float)
    finite_mask = np.isfinite(numeric_array)
    return numeric_array, finite_mask

def _is_line_trace_for_current_value(trace):
    if _is_current_value_reference_trace(trace):
        return False
    if getattr(trace, "type", None) not in {"scatter", "scattergl"}:
        return False
    if "lines" not in str(getattr(trace, "mode", "")):
        return False
    if getattr(trace, "hoverinfo", None) == "skip":
        return False
    fill = getattr(trace, "fill", None)
    if fill not in (None, "none"):
        return False

    numeric_y, finite_mask = _numeric_trace_y(trace)
    if numeric_y is None or finite_mask.sum() < 2:
        return False
    return np.unique(numeric_y[finite_mask]).size > 1

def _trace_current_value_reference_payload(trace):
    numeric_y, finite_mask = _numeric_trace_y(trace)
    if numeric_y is None or finite_mask.sum() < 2:
        return None

    x_values = getattr(trace, "x", None)
    if x_values is None:
        x_values = list(range(len(numeric_y)))
    else:
        x_values = list(x_values)
    if len(x_values) != len(numeric_y):
        x_values = list(range(len(numeric_y)))

    valid_x = [x_value for x_value, is_valid in zip(x_values, finite_mask) if is_valid]
    current_value = numeric_y[finite_mask][-1]
    return valid_x[0], valid_x[-1], current_value

def _extend_visibility_buttons_for_current_value_lines(fig, source_indices):
    if not source_indices:
        return
    original_trace_count = len(fig.data) - len(source_indices)
    for menu in fig.layout.updatemenus or []:
        for button in menu.buttons or []:
            args = list(button.args or [])
            if not args or not isinstance(args[0], dict) or "visible" not in args[0]:
                continue
            visible = list(args[0]["visible"])
            if len(visible) != original_trace_count:
                continue
            args[0]["visible"] = visible + [
                _current_value_reference_visibility(visible[source_index])
                for source_index in source_indices
            ]
            button.args = tuple(args)

def _style_current_value_reference_lines(fig):
    for trace in fig.data:
        if _is_current_value_reference_trace(trace):
            trace.update(line=CURRENT_VALUE_REFERENCE_LINE_STYLE.copy(), visible="legendonly", showlegend=True)

def _add_current_value_reference_lines(fig):
    if any(_is_current_value_reference_trace(trace) for trace in fig.data):
        _style_current_value_reference_lines(fig)
        return fig

    source_indices = []
    for source_index, trace in enumerate(list(fig.data)):
        if not _is_line_trace_for_current_value(trace):
            continue
        payload = _trace_current_value_reference_payload(trace)
        if payload is None:
            continue
        x_start, x_end, current_value = payload
        reference_trace = dict(
            x=[x_start, x_end],
            y=[current_value, current_value],
            mode="lines",
            name=f"{getattr(trace, 'name', '') or 'Series'} Current Value",
            line=CURRENT_VALUE_REFERENCE_LINE_STYLE.copy(),
            hoverinfo="skip",
            showlegend=True,
            visible=_current_value_reference_visibility(getattr(trace, "visible", True)),
            meta={CURRENT_VALUE_REFERENCE_META_KEY: True},
        )
        xaxis = getattr(trace, "xaxis", None)
        yaxis = getattr(trace, "yaxis", None)
        if xaxis:
            reference_trace["xaxis"] = xaxis
        if yaxis:
            reference_trace["yaxis"] = yaxis
        fig.add_trace(go.Scatter(**reference_trace))
        source_indices.append(source_index)

    _extend_visibility_buttons_for_current_value_lines(fig, source_indices)
    return fig

if not hasattr(go.Figure, "_quantapp_original_show"):
    go.Figure._quantapp_original_show = go.Figure.show

def _quantapp_show_with_current_value_lines(self, *args, **kwargs):
    _add_current_value_reference_lines(self)
    return go.Figure._quantapp_original_show(self, *args, **kwargs)

go.Figure.show = _quantapp_show_with_current_value_lines

In [ ]:
# Block 4: load shared parameters and set notebook-specific controls

pricing_params = get_single_asset_params()
ticker_str = pricing_params["ticker_str"]
interval = pricing_params["interval"]
period = pricing_params["period"]

vix_str = "^VIX"
risk_free_ticker = "^IRX"
benchmark_tickers = ["SPY"]
include_factor_peer_index = True
auto_build_factor_peer_index = True  # Build a missing GICS peer index inside this notebook.
factor_peer_index_max_symbols = 40  # Limit peer downloads while retaining a broad peer basket.
factor_peer_index_source_ticker = None  # None = current ticker. For an ETF, optionally set a representative stock whose Factor peer index should be used.
peer_index_cache_dir = PROJECT_ROOT / "company_data" / "factor_peer_indexes"
time_frame_map = {"short": 21, "mid": 50, "long": 200}
selected_time_frames = [21, 50, 200]
default_window = 200
length_of_plots = 20
var_position_value = None

In [ ]:
# Block 4A: Direct yfinance Price Retrieval Check
# Run this after Block 4 when you want to verify raw Yahoo Finance prices.

import yfinance as yf

direct_yfinance_symbols = list(dict.fromkeys([
    ticker_str,
    vix_str,
    risk_free_ticker,
    *benchmark_tickers,
]))

def _direct_yfinance_flatten_columns(frame):
    if not isinstance(frame.columns, pd.MultiIndex):
        return frame

    standard_price_columns = {"Open", "High", "Low", "Close", "Adj Close", "Volume"}
    for level in range(frame.columns.nlevels):
        level_values = pd.Index(frame.columns.get_level_values(level))
        if standard_price_columns.intersection(set(level_values.astype(str))):
            flattened = frame.copy()
            flattened.columns = level_values
            return flattened.loc[:, ~flattened.columns.duplicated()]

    flattened = frame.copy()
    flattened.columns = ["_".join(str(part) for part in column if str(part)) for column in frame.columns.to_flat_index()]
    return flattened

def _direct_yfinance_history(symbol):
    frame = yf.download(
        symbol,
        period=period,
        interval=interval,
        auto_adjust=False,
        progress=False,
        threads=False,
    )
    if frame.empty:
        frame = yf.Ticker(symbol).history(
            period=period,
            interval=interval,
            auto_adjust=False,
        )
    frame = _direct_yfinance_flatten_columns(frame)
    if not frame.empty:
        frame = frame.copy()
        frame.index = pd.to_datetime(frame.index, errors="coerce").tz_localize(None).normalize()
        frame = frame[~frame.index.isna()].sort_index()
    return frame

direct_yfinance_price_history = {
    symbol: _direct_yfinance_history(symbol)
    for symbol in direct_yfinance_symbols
}

direct_yfinance_price_summary = pd.DataFrame(
    [
        {
            "Symbol": symbol,
            "Rows": len(frame),
            "Valid Close Rows": int(pd.to_numeric(frame.get("Close", pd.Series(dtype=float)), errors="coerce").notna().sum()) if not frame.empty else 0,
            "Start": frame.index.min() if not frame.empty else pd.NaT,
            "End": frame.index.max() if not frame.empty else pd.NaT,
            "Last Close": pd.to_numeric(frame.get("Close", pd.Series(dtype=float)), errors="coerce").dropna().iloc[-1] if not frame.empty and not pd.to_numeric(frame.get("Close", pd.Series(dtype=float)), errors="coerce").dropna().empty else np.nan,
        }
        for symbol, frame in direct_yfinance_price_history.items()
    ]
)

#display(direct_yfinance_price_summary)
#direct_yfinance_price_history.get(ticker_str, pd.DataFrame()).tail()


In [ ]:
# Block 5: fetch market history and assign notebook roles
requested_symbols = [
    ticker_str,
    vix_str,
    risk_free_ticker,
    *benchmark_tickers,
]

asset_histories = get_market_history(
    symbols=requested_symbols,
    period=period,
    interval=interval,
    provider="yfinance",
    # Keep histories unaligned here so a sparse proxy such as ^IRX cannot collapse
    # the asset and benchmark histories to only their shared dates.
    align=False,
)

asset_history           = asset_histories.get(ticker_str, pd.DataFrame())
vix_history             = asset_histories.get(vix_str, pd.DataFrame())
risk_free_proxy_history = asset_histories.get(risk_free_ticker, pd.DataFrame())

benchmark_data = {
    symbol: frame
    for symbol, frame in asset_histories.items()
    if symbol not in {ticker_str, vix_str, risk_free_ticker}
}

factor_gics_index_suffixes = {
    "Sector": "sector",
    "Industry Group": "industry_group",
    "Industry": "industry",
    "Sub-Industry": "sub_industry",
}

def load_factor_index_cache(cache_path, default_label):
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return None, None
    cached = pd.read_csv(cache_path)
    required_columns = {"Date", "Close"}
    if not required_columns.issubset(cached.columns):
        raise ValueError(f"Factor-index cache is missing columns {sorted(required_columns - set(cached.columns))}: {cache_path}")

    dates = pd.to_datetime(cached["Date"], errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()
    close = pd.to_numeric(cached["Close"], errors="coerce")
    index_frame = pd.DataFrame({"Close": close.to_numpy()}, index=dates)
    index_frame = index_frame.loc[~index_frame.index.isna()].dropna(subset=["Close"])
    index_frame = index_frame.loc[~index_frame.index.duplicated(keep="last")].sort_index()
    if not asset_history.empty:
        index_frame = index_frame.loc[asset_history.index.min():asset_history.index.max()]

    cached_label = cached.get("Benchmark Label", pd.Series(dtype="object")).dropna()
    label = str(cached_label.iloc[-1]) if not cached_label.empty else default_label
    cached_level = cached.get("GICS Level", pd.Series(dtype="object")).dropna()
    level = str(cached_level.iloc[-1]) if not cached_level.empty else None
    return index_frame, (label, level)

def load_factor_peer_index(target_symbol, cache_dir):
    safe_symbol = str(target_symbol).replace(".", "_").replace("/", "_")
    cache_path = Path(cache_dir) / f"{safe_symbol}_peer_index.csv"
    peer_frame, metadata = load_factor_index_cache(cache_path, f"{target_symbol} Peer Index")
    return peer_frame, cache_path, metadata

def build_missing_factor_peer_index(target_symbol, cache_dir, max_symbols=40):
    """Build and cache an equal-weight GICS sub-industry peer index."""
    company_cache_path = PROJECT_ROOT / "company_data" / "gics_companies.csv"
    if company_cache_path.exists():
        companies = pd.read_csv(company_cache_path)
    else:
        gics_client = GICSDataClient(save_path=PROJECT_ROOT)
        companies = gics_client.retrieve_companies()
        company_cache_path.parent.mkdir(parents=True, exist_ok=True)
        companies.to_csv(company_cache_path, index=False)

    peer_context = build_gics_peer_frames(
        target_symbol, companies=companies, symbol_label="YFinance Symbol"
    )
    peer_rows = peer_context.frame("Sub-Industry")
    peer_symbols = list(dict.fromkeys(peer_rows["Normalized Symbol"].dropna().astype(str)))
    peer_symbols = [symbol for symbol in peer_symbols if symbol != peer_context.target_symbol][:max_symbols]
    if len(peer_symbols) < 2:
        raise ValueError(
            f"At least two GICS sub-industry peers are required for {target_symbol}; found {len(peer_symbols)}."
        )

    peer_histories = get_market_history(
        symbols=peer_symbols, period=period, interval=interval, provider="yfinance", align=False
    )
    peer_closes = []
    for symbol in peer_symbols:
        history = peer_histories.get(symbol, pd.DataFrame())
        if history is not None and not history.empty and "Close" in history.columns:
            peer_closes.append(pd.to_numeric(history["Close"], errors="coerce").rename(symbol))
    if len(peer_closes) < 2:
        raise ValueError(f"Usable price history was available for only {len(peer_closes)} peers.")

    peer_prices = pd.concat(peer_closes, axis=1).sort_index()
    peer_returns = peer_prices.pct_change(fill_method=None)
    valid_counts = peer_returns.notna().sum(axis=1)
    index_returns = peer_returns.mean(axis=1, skipna=True).where(valid_counts >= 2).dropna()
    if index_returns.empty:
        raise ValueError(f"Peer histories for {target_symbol} did not have overlapping return dates.")
    peer_index = (100.0 * (1.0 + index_returns).cumprod()).rename("Close")
    first_date = peer_prices.index[peer_prices.index < peer_index.index[0]]
    if len(first_date):
        peer_index = pd.concat([pd.Series([100.0], index=first_date[-1:], name="Close"), peer_index])

    safe_symbol = str(target_symbol).replace(".", "_").replace("/", "_")
    cache_path = Path(cache_dir) / f"{safe_symbol}_peer_index.csv"
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    peer_label = f"{target_symbol} Sub-Industry Equal-Weight Peer Index"
    export_frame = peer_index.to_frame()
    export_frame["Target Symbol"] = target_symbol
    export_frame["Benchmark Label"] = peer_label
    export_frame["GICS Level"] = "Sub-Industry"
    export_frame["GICS Name"] = peer_context.target_row["Sub-Industry"]
    export_frame["Weighting"] = "Equal Weight"
    export_frame["Constituent Count"] = len(peer_closes)
    export_frame.to_csv(cache_path, index_label="Date")
    return cache_path, peer_label, len(peer_closes)

def load_factor_gics_indexes(target_symbol, cache_dir):
    safe_symbol = str(target_symbol).replace(".", "_").replace("/", "_")
    loaded_indexes = {}
    cache_paths = {}
    for gics_level, file_suffix in factor_gics_index_suffixes.items():
        cache_path = Path(cache_dir) / f"{safe_symbol}_{file_suffix}_index.csv"
        cache_paths[gics_level] = cache_path
        index_frame, metadata = load_factor_index_cache(
            cache_path, f"{target_symbol} {gics_level} Index"
        )
        if index_frame is not None:
            label, stored_level = metadata
            loaded_indexes[label] = {
                "frame": index_frame,
                "level": stored_level or gics_level,
                "path": cache_path,
            }
    return loaded_indexes, cache_paths

if include_factor_peer_index:
    factor_peer_lookup_symbol = factor_peer_index_source_ticker or ticker_str
    factor_gics_benchmarks, factor_gics_cache_paths = load_factor_gics_indexes(
        factor_peer_lookup_symbol, peer_index_cache_dir
    )
    if factor_gics_benchmarks:
        for factor_index_label, factor_index_payload in factor_gics_benchmarks.items():
            factor_index_frame = factor_index_payload["frame"]
            if factor_index_frame.empty:
                print(f"Factor index cache has no dates overlapping the asset history: {factor_index_payload['path']}")
                continue
            benchmark_data[factor_index_label] = factor_index_frame
            print(
                f"Loaded benchmark {factor_index_label} ({factor_index_payload['level']})"
                + (f" for ETF/asset {ticker_str}" if factor_peer_lookup_symbol != ticker_str else "")
                + f" from {factor_index_payload['path']}"
            )
        missing_gics_levels = [
            level for level, path in factor_gics_cache_paths.items() if not path.exists()
        ]
        if missing_gics_levels:
            print(f"Missing Factor GICS index caches: {missing_gics_levels}. Rerun Factor Analysis block 5.")
    else:
        factor_peer_frame, factor_peer_cache_path, factor_peer_metadata = load_factor_peer_index(
            factor_peer_lookup_symbol, peer_index_cache_dir
        )
        if factor_peer_frame is not None and not factor_peer_frame.empty:
            factor_peer_label, factor_peer_level = factor_peer_metadata
            benchmark_data[factor_peer_label] = factor_peer_frame
            print(
                f"Loaded legacy benchmark {factor_peer_label}"
                + (f" ({factor_peer_level})" if factor_peer_level else "")
                + f" from {factor_peer_cache_path}"
            )
        elif auto_build_factor_peer_index:
            try:
                built_path, built_label, constituent_count = build_missing_factor_peer_index(
                    factor_peer_lookup_symbol,
                    peer_index_cache_dir,
                    max_symbols=factor_peer_index_max_symbols,
                )
                factor_peer_frame, _, factor_peer_metadata = load_factor_peer_index(
                    factor_peer_lookup_symbol, peer_index_cache_dir
                )
                if factor_peer_frame is None or factor_peer_frame.empty:
                    raise ValueError("The generated peer index has no dates overlapping the asset history.")
                benchmark_data[built_label] = factor_peer_frame
                print(
                    f"Built and loaded {built_label} from {constituent_count} GICS peers; "
                    f"cached at {built_path}"
                )
            except Exception as exc:
                print(
                    f"Could not automatically build a peer index for {factor_peer_lookup_symbol}: {exc}. "
                    f"Using {benchmark_tickers} only. For an ETF, set "
                    "factor_peer_index_source_ticker to a representative company."
                )
        else:
            print(
                f"Peer index not found for {factor_peer_lookup_symbol}; using {benchmark_tickers} only."
            )

#loaded_benchmark_tickers = list(benchmark_data)
#analysis_index = asset_history.index


In [ ]:
# Block 7: derive analysis series from normalized market data

try:
    risk_free_daily_rate = series_transforms.annualized_yield_to_periodic_rate(
        risk_free_proxy_history,
        annualization_factor=252,
        input_is_percent=True,
        lag_periods=1,
        reference_index=asset_history.index,
    )
except (TypeError, ValueError):
    risk_free_daily_rate = pd.Series(0.0, index=asset_history.index)

risk_free_daily_rate = pd.Series(risk_free_daily_rate, index=asset_history.index).replace([np.inf, -np.inf], np.nan)
if risk_free_daily_rate.dropna().empty:
    if isinstance(risk_free_proxy_history, pd.DataFrame) and "Close" in risk_free_proxy_history:
        latest_risk_free_yield = pd.to_numeric(risk_free_proxy_history["Close"], errors="coerce").dropna()
    else:
        latest_risk_free_yield = pd.Series(dtype=float)

    if latest_risk_free_yield.empty:
        risk_free_daily_rate = pd.Series(0.0, index=asset_history.index)
        print("Risk-free proxy unavailable; using 0.00% annual risk-free rate fallback.")
    else:
        fallback_annual_rate = float(latest_risk_free_yield.iloc[-1]) / 100.0
        fallback_daily_rate = (1.0 + fallback_annual_rate) ** (1.0 / 252) - 1.0
        risk_free_daily_rate = pd.Series(fallback_daily_rate, index=asset_history.index)
        print(
            f"Risk-free proxy has sparse aligned data; using latest {risk_free_ticker} "
            f"yield as a constant fallback: {fallback_annual_rate:.2%} annual."
        )
else:
    risk_free_daily_rate = risk_free_daily_rate.ffill().bfill()


ticker_monthly_data = series_transforms.resample(asset_history, frequency="monthly")
ticker_weekly_data = series_transforms.resample(asset_history, frequency="weekly")
ticker_daily_data = series_transforms.resample(asset_history, frequency="daily")

ticker_monthly_returns = ticker_monthly_data["Close"].pct_change(fill_method=None).dropna()
ticker_weekly_returns = ticker_weekly_data["Close"].pct_change(fill_method=None).dropna()
ticker_daily_returns = ticker_daily_data["Close"].pct_change(fill_method=None).dropna()


In [ ]:
# Block 12: plot stacked rolling Sharpe z-score heatmaps for 1-200 day windows plus cross-window summaries

heatmap_windows = list(range(1, 201))

def heatmap_zscore(series):
    clean = pd.Series(series).dropna().sort_index()
    if clean.empty:
        return pd.Series(dtype=float)
    std = clean.std()
    if std == 0 or pd.isna(std):
        return pd.Series(0.0, index=clean.index)
    return (clean - clean.mean()) / std

def rolling_sharpe_frame(close):
    return risk_adjusted_returns(
        close.dropna().sort_index(),
        windows=heatmap_windows,
        ratio_type="sharpe",
        risk_free_rate=risk_free_daily_rate,
    ).set_axis(heatmap_windows, axis=1)

asset_sharpe_frame = rolling_sharpe_frame(asset_history["Close"])

asset_sharpe_zscore_frame = asset_sharpe_frame.apply(heatmap_zscore)

benchmark_sharpe_zscore_frames = {}
benchmark_spread_zscore_frames = {}
for symbol, benchmark_frame in benchmark_data.items():
    benchmark_sharpe_frame = rolling_sharpe_frame(benchmark_frame["Close"])
    benchmark_sharpe_zscore_frames[symbol] = benchmark_sharpe_frame.apply(heatmap_zscore)
    benchmark_spread = benchmark_sharpe_frame - asset_sharpe_frame
    benchmark_spread_zscore_frames[symbol] = benchmark_spread.apply(heatmap_zscore)


#display(benchmark_spread_zscore_frames)
fig_block12_sharpe_zscore_heatmap = plot_sharpe_zscore_heatmap_view(
    asset_sharpe_zscore_frame=asset_sharpe_zscore_frame,
    benchmark_sharpe_zscore_frames=benchmark_sharpe_zscore_frames,
    benchmark_spread_zscore_frames=benchmark_spread_zscore_frames,
    ticker_label=ticker_str,
)
fig = fig_block12_sharpe_zscore_heatmap


In [ ]:
# Block 11: compute rolling Sharpe windows, momentum histograms, and volatility

block11_min_window = 3
block11_fallback_max_window = 400
annualization_factor = 252

import yfinance as yf

def block11_current_option_chain_dtes(symbol, *, min_dte=0, max_dte=None):
    try:
        expiration_values = yf.Ticker(symbol).options or []
    except Exception as error:
        print(f"Option-chain expirations unavailable for {symbol}: {error}")
        return pd.DataFrame(columns=["Expiration Date", "DTE"])

    expiration_dates = pd.to_datetime(list(expiration_values), errors="coerce")
    expiration_dates = pd.DatetimeIndex(expiration_dates).dropna().normalize()
    if expiration_dates.empty:
        return pd.DataFrame(columns=["Expiration Date", "DTE"])

    today = pd.Timestamp.today().normalize()
    dte_frame = pd.DataFrame({"Expiration Date": expiration_dates})
    dte_frame["DTE"] = (dte_frame["Expiration Date"] - today).dt.days.astype(int)
    dte_frame = dte_frame[dte_frame["DTE"] >= int(min_dte)]
    if max_dte is not None:
        dte_frame = dte_frame[dte_frame["DTE"] <= int(max_dte)]
    return dte_frame.drop_duplicates("DTE").sort_values("DTE").reset_index(drop=True)

def block11_add_option_dte_vlines(fig, dte_frame):
    if dte_frame.empty:
        print(
            f"No listed {ticker_str} option-chain DTEs fall inside the "
            f"{min(window_sizes)}-{max(window_sizes)} day Block 11 window range."
        )
        return fig

    marker_rows = [
        (1, 2), (2, 1), (2, 2),
        (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1),
    ]
    annotation_rows = {(1, 2), (2, 1), (2, 2), (3, 1), (6, 1)}
    for marker_index, marker in dte_frame.iterrows():
        dte = int(marker["DTE"])
        expiration_date = pd.Timestamp(marker["Expiration Date"])
        for subplot_row, subplot_col in marker_rows:
            vline_kwargs = dict(
                x=dte,
                line_color="rgba(250, 204, 21, 0.58)",
                line_width=1,
                line_dash="dot",
                opacity=0.72,
                row=subplot_row,
                col=subplot_col,
            )
            if (subplot_row, subplot_col) in annotation_rows:
                vline_kwargs.update(
                    annotation_text=f"Exact {dte} DTE<br>{expiration_date:%Y-%m-%d}",
                    annotation_position="top left",
                    annotation_font_size=10,
                    annotation_font_color="rgba(250, 204, 21, 0.92)",
                )
            fig.add_vline(**vline_kwargs)
    return fig

block11_option_chain_dtes = block11_current_option_chain_dtes(
    ticker_str,
    min_dte=block11_min_window,
    max_dte=None,
)
if block11_option_chain_dtes.empty:
    block11_max_window = block11_fallback_max_window
    block11_window_limit_source = "fallback because the option chain was unavailable"
    print(
        f"No current option-chain expirations were available for {ticker_str}; "
        f"using the {block11_fallback_max_window}-day diagnostics fallback."
    )
else:
    block11_max_window = int(block11_option_chain_dtes["DTE"].max())
    block11_window_limit_source = f"latest listed {ticker_str} option DTE"
window_sizes = list(range(block11_min_window, block11_max_window + 1))
print(
    f"Block 11 diagnostics windows: {block11_min_window}-{block11_max_window} days "
    f"({block11_window_limit_source})."
)

def build_momentum_diagnostics_context(close, *, highlight_windows=()):
    momentum_close = pd.to_numeric(close, errors="coerce").dropna().sort_index()
    aligned_risk_free_daily_rate = pd.Series(risk_free_daily_rate, index=momentum_close.index).ffill().bfill()

    sharpe_table = risk_adjusted_returns(
        momentum_close,
        windows=window_sizes,
        ratio_type="sharpe",
        risk_free_rate=aligned_risk_free_daily_rate,
        annualization_factor=annualization_factor,
    ).set_axis(window_sizes, axis=1).replace([np.inf, -np.inf], np.nan).dropna(how="all").copy()

    returns = momentum_close.pct_change()
    excess_returns = returns - aligned_risk_free_daily_rate
    volatility_df = np.sqrt(annualization_factor) * compute.rolling_windows(
        excess_returns,
        metric=pd.Series.std,
        windows=window_sizes,
    )

    return {
        "sharpe_table": sharpe_table,
        "volatility_df": volatility_df,
        "highlight_windows": tuple(highlight_windows),
    }

from Quantapp.visualization.views.single_asset_profile.pricing.momentum_efficiency._shared import coerce_momentum_diagnostics_context
from Quantapp.visualization.views.single_asset_profile.pricing.momentum_efficiency.momentum_window_diagnostics_grid import _horizon_derivative_series

def block11_axis_range(series_collection, *, padding=0.12, min_span=1.0):
    finite_chunks = []
    for values in series_collection:
        numeric = pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)
        finite_values = numeric[np.isfinite(numeric)]
        if finite_values.size > 0:
            finite_chunks.append(finite_values)
    if not finite_chunks:
        return None
    values = np.concatenate(finite_chunks)
    lower = float(values.min())
    upper = float(values.max())
    span = upper - lower
    if span <= 0:
        midpoint = (lower + upper) / 2.0
        span = min_span
        lower = midpoint - span / 2.0
        upper = midpoint + span / 2.0
    axis_padding = max(span * padding, min_span * 0.05)
    return [lower - axis_padding, upper + axis_padding]

def block11_reference_band_series(mean_by_window, std_by_window):
    mean_series = pd.Series(mean_by_window)
    std_series = pd.Series(std_by_window)
    return [
        mean_series + sign * std_series * level
        for level in (1, 2)
        for sign in (1, -1)
    ]

def block11_add_horizon_derivative_traces(fig, series, *, symbol, base_name, color, dash, rows):
    for order, row in zip((1, 2), rows):
        derivative = _horizon_derivative_series(series.index, series.values, order=order)
        derivative_label = "First Derivative" if order == 1 else "Second Derivative"
        units = "Z-score / 20 horizon days" if order == 1 else "Z-score / (20 horizon days)^2"
        fig.add_trace(
            go.Scatter(
                x=derivative.index,
                y=derivative.values,
                mode="lines",
                name=f"{base_name} — {derivative_label}",
                line=dict(color=color, width=2.0, dash=dash),
                showlegend=False,
                hovertemplate=(
                    f"Benchmark / index: {symbol}<br>"
                    "Lookback horizon: %{x} trading days<br>"
                    + derivative_label + ": %{y:.3f} " + units + "<extra></extra>"
                ),
            ),
            row=row,
            col=1,
        )

momentum_diagnostics_contexts = {
    ticker_str: build_momentum_diagnostics_context(asset_history["Close"]),
}

for symbol, benchmark_frame in benchmark_data.items():
    if isinstance(benchmark_frame, pd.DataFrame) and "Close" in benchmark_frame:
        momentum_diagnostics_contexts[symbol] = build_momentum_diagnostics_context(benchmark_frame["Close"])

# Preserve the original single-asset names for downstream notebook cells.
momentum_diagnostics_context = momentum_diagnostics_contexts[ticker_str]
sharpe_table = momentum_diagnostics_context["sharpe_table"]
volatility_df = momentum_diagnostics_context["volatility_df"]

momentum_diagnostics_display_contexts = {
    symbol: coerce_momentum_diagnostics_context(context)
    for symbol, context in momentum_diagnostics_contexts.items()
}

fig_momentum_window_diagnostics_grid = plot_momentum_window_diagnostics_grid_view(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)

block11_benchmark_colors = ["#f97316", "#22c55e", "#facc15", "#ef4444", "#ec4899", "#f8fafc"]
block11_benchmark_dashes = ["dash", "dot", "longdash", "dashdot", "solid"]

for benchmark_index, (symbol, display_context) in enumerate(momentum_diagnostics_display_contexts.items()):
    if symbol == ticker_str:
        continue
    color = block11_benchmark_colors[benchmark_index % len(block11_benchmark_colors)]
    dash = block11_benchmark_dashes[benchmark_index % len(block11_benchmark_dashes)]

    benchmark_current_sharpe_zscore = display_context["current_sharpe_zscore"].dropna()
    if not benchmark_current_sharpe_zscore.empty:
        fig_momentum_window_diagnostics_grid.add_trace(
            go.Scatter(
                x=benchmark_current_sharpe_zscore.index,
                y=benchmark_current_sharpe_zscore.values,
                mode="lines+markers",
                name=f"{symbol} Current Sharpe Z-Score",
                line=dict(color=color, width=2.2, dash=dash),
                marker=dict(size=4),
                hovertemplate=(
                    "Benchmark: " + symbol + "<br>"
                    "Window: %{x} day(s)<br>"
                    "Current Sharpe Z-Score: %{y:.2f}<extra></extra>"
                ),
            ),
            row=3,
            col=1,
        )
        block11_add_horizon_derivative_traces(
            fig_momentum_window_diagnostics_grid,
            benchmark_current_sharpe_zscore,
            symbol=symbol,
            base_name=f"{symbol} Current Sharpe Z-Score",
            color=color,
            dash=dash,
            rows=(4, 5),
        )

    benchmark_cross_window_zscore = display_context["current_sharpe_cross_window_zscore"].dropna()
    if not benchmark_cross_window_zscore.empty:
        fig_momentum_window_diagnostics_grid.add_trace(
            go.Scatter(
                x=benchmark_cross_window_zscore.index,
                y=benchmark_cross_window_zscore.values,
                mode="lines+markers",
                name=f"{symbol} Cross-Window Relative Z-Score",
                line=dict(color=color, width=2.2, dash=dash),
                marker=dict(size=4),
                hovertemplate=(
                    "Benchmark: " + symbol + "<br>"
                    "Window: %{x} day(s)<br>"
                    "Cross-Window Relative Z-Score: %{y:.2f}<extra></extra>"
                ),
            ),
            row=6,
            col=1,
        )
        block11_add_horizon_derivative_traces(
            fig_momentum_window_diagnostics_grid,
            benchmark_cross_window_zscore,
            symbol=symbol,
            base_name=f"{symbol} Cross-Window Relative Z-Score",
            color=color,
            dash=dash,
            rows=(7, 8),
        )

asset_display_context = momentum_diagnostics_display_contexts[ticker_str]
row3_range_series = [
    asset_display_context["current_sharpe_zscore"],
    asset_display_context["sharpe_zscore_mean_by_window"],
    *block11_reference_band_series(
        asset_display_context["sharpe_zscore_mean_by_window"],
        asset_display_context["sharpe_zscore_std_by_window"],
    ),
    *[
        context["current_sharpe_zscore"]
        for symbol, context in momentum_diagnostics_display_contexts.items()
        if symbol != ticker_str
    ],
    pd.Series([-2.0, 2.0]),
]
row4_range_series = [
    asset_display_context["current_sharpe_cross_window_zscore"],
    asset_display_context["cross_window_zscore_mean_by_window"],
    *block11_reference_band_series(
        asset_display_context["cross_window_zscore_mean_by_window"],
        asset_display_context["cross_window_zscore_std_by_window"],
    ),
    *[
        context["current_sharpe_cross_window_zscore"]
        for symbol, context in momentum_diagnostics_display_contexts.items()
        if symbol != ticker_str
    ],
    pd.Series([-2.0, 2.0]),
]

fig_momentum_window_diagnostics_grid.update_yaxes(range=block11_axis_range(row3_range_series), row=3, col=1)
fig_momentum_window_diagnostics_grid.update_yaxes(range=block11_axis_range(row4_range_series), row=6, col=1)
block11_add_option_dte_vlines(fig_momentum_window_diagnostics_grid, block11_option_chain_dtes)
fig = fig_momentum_window_diagnostics_grid


In [ ]:
# Block 8: compute rolling arithmetic/geometric mean returns with MAD-score details

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from Quantapp.visualization.views.single_asset_profile.pricing.momentum_efficiency._shared import (
    finalize_dark_figure,
    header_margin,
    header_title,
    trace_datetime_bounds,
)

configured_rolling_mean_windows = globals().get(
    "selected_time_frames",
    [globals().get("default_window", 200)],
)
if isinstance(configured_rolling_mean_windows, (int, float, np.integer)):
    configured_rolling_mean_windows = [configured_rolling_mean_windows]

rolling_mean_windows = []
for window in configured_rolling_mean_windows:
    try:
        window = int(window)
    except (TypeError, ValueError):
        continue
    if window > 0 and window not in rolling_mean_windows:
        rolling_mean_windows.append(window)
if not rolling_mean_windows:
    rolling_mean_windows = [200]

preferred_rolling_mean_window = int(globals().get("default_window", 200))
rolling_mean_window = (
    preferred_rolling_mean_window
    if preferred_rolling_mean_window in rolling_mean_windows
    else 200
    if 200 in rolling_mean_windows
    else max(rolling_mean_windows)
)

def rolling_arithmetic_label(window):
    return f"{int(window)}D Arithmetic Mean"

def rolling_geometric_label(window):
    return f"{int(window)}D Geometric Mean"

compounding_efficiency_label = "Compounding Efficiency"
volatility_drag_label = "Volatility Drag"

def compounding_efficiency_from_means(arithmetic_mean, geometric_mean):
    return (1.0 + geometric_mean).div(1.0 + arithmetic_mean).replace([np.inf, -np.inf], np.nan)

def volatility_drag_from_means(arithmetic_mean, geometric_mean):
    return arithmetic_mean - geometric_mean

def rolling_metric_mad_score(series):
    clean = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna().sort_index()
    if clean.empty:
        return pd.Series(dtype=float)
    median = clean.median()
    mad = (clean - median).abs().median()
    if mad == 0 or pd.isna(mad):
        return pd.Series(0.0, index=clean.index)
    return ((clean - median) / (1.4826 * mad)).dropna()

def metric_detail_array(value_series, mad_score_series, index):
    detail_frame = pd.concat(
        {
            "value": pd.Series(value_series),
            "mad_score": pd.Series(mad_score_series),
        },
        axis=1,
    ).reindex(index)
    return detail_frame[["value", "mad_score"]].to_numpy()

def mad_score_annotation_text(metric_label, metric_symbol, latex_formula, function_name):
    return (
        f"<b>{metric_label}</b><br>"
        f"${metric_symbol}_t = {latex_formula}$<br>"
        rf"$m_t = \frac{{{metric_symbol}_t - \operatorname{{median}}({metric_symbol})}}{{1.4826 \cdot \operatorname{{MAD}}({metric_symbol})}}$<br>"
        rf"$\operatorname{{MAD}}({metric_symbol}) = \operatorname{{median}}(|{metric_symbol} - \operatorname{{median}}({metric_symbol})|)$<br>"
        f"<span style=\"font-family:monospace;font-size:10px\">{function_name}</span>"
    )

def add_formula_annotation(fig, text, y_position):
    fig.add_annotation(
        text=text,
        x=0.01,
        y=y_position,
        xref="paper",
        yref="paper",
        xanchor="left",
        yanchor="top",
        showarrow=False,
        align="left",
        bgcolor="rgba(15, 23, 42, 0.72)",
        bordercolor="rgba(148, 163, 184, 0.34)",
        borderwidth=1,
        font=dict(size=11, color="rgba(226, 232, 240, 0.95)"),
    )

def block8_dropdown_menu(buttons, x, active=0):
    return dict(
        type="dropdown",
        buttons=buttons,
        direction="down",
        showactive=True,
        active=active,
        x=x,
        xanchor="left",
        y=1.09,
        yanchor="top",
        bgcolor="rgba(15, 23, 42, 0.92)",
        bordercolor="rgba(148, 163, 184, 0.35)",
        font=dict(color="rgba(226, 232, 240, 0.96)"),
    )

def block8_time_range_buttons(global_start, global_end, axis_count):
    def make_range(years=None):
        start = global_start if years is None else max(global_start, global_end - pd.DateOffset(years=years))
        return {
            ("xaxis.range" if axis_idx == 1 else f"xaxis{axis_idx}.range"): [start, global_end]
            for axis_idx in range(1, axis_count + 1)
        }

    return [
        dict(label="10 Years", method="relayout", args=[make_range(10)]),
        dict(label="5 Years", method="relayout", args=[make_range(5)]),
        dict(label="3 Years", method="relayout", args=[make_range(3)]),
        dict(label="1 Year", method="relayout", args=[make_range(1)]),
        dict(label="All", method="relayout", args=[make_range(None)]),
    ]

def _block8_clean_return_series(series):
    return (
        pd.to_numeric(pd.Series(series), errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .sort_index()
    )

if "ticker_daily_returns" not in globals():
    if "ticker_daily_data" in globals() and isinstance(ticker_daily_data, pd.DataFrame) and "Close" in ticker_daily_data:
        ticker_daily_returns = ticker_daily_data["Close"].pct_change(fill_method=None).dropna()
    elif "asset_history" in globals() and isinstance(asset_history, pd.DataFrame) and "Close" in asset_history:
        ticker_daily_returns = asset_history["Close"].pct_change(fill_method=None).dropna()
    else:
        raise ValueError("Block 8 needs ticker_daily_returns from Block 7. Run Blocks 4, 5, and 7 first.")

rolling_mean_returns = _block8_clean_return_series(ticker_daily_returns)
if rolling_mean_returns.empty:
    asset_rows = len(asset_history) if "asset_history" in globals() and isinstance(asset_history, pd.DataFrame) else 0
    daily_rows = len(ticker_daily_data) if "ticker_daily_data" in globals() and isinstance(ticker_daily_data, pd.DataFrame) else 0
    raise ValueError(
        "Block 8 has no clean daily returns. This usually means Block 5 did not retrieve usable price data, "
        "Block 7 was not rerun after changing parameters, or the Close series is empty. "
        f"asset_history rows: {asset_rows:,}; ticker_daily_data rows: {daily_rows:,}. "
        "Run Blocks 4, 5, and 7 before Block 8."
    )

available_return_count = len(rolling_mean_returns)
requested_rolling_mean_windows = list(rolling_mean_windows)
rolling_mean_windows = [window for window in rolling_mean_windows if window <= available_return_count]
skipped_rolling_mean_windows = [
    window for window in requested_rolling_mean_windows if window > available_return_count
]
if not rolling_mean_windows:
    if available_return_count < 2:
        raise ValueError(
            "Block 8 needs at least two clean daily returns for a rolling mean plot. "
            f"Only {available_return_count:,} clean return row(s) are available. "
            "Run Blocks 4, 5, and 7 and confirm the selected ticker retrieved price history."
        )
    fallback_window = min(max(2, available_return_count), preferred_rolling_mean_window)
    rolling_mean_windows = [fallback_window]
    print(
        "Block 8: requested rolling windows "
        f"{requested_rolling_mean_windows} are larger than the {available_return_count:,} "
        f"available clean daily returns. Using {fallback_window}D instead."
    )
elif skipped_rolling_mean_windows:
    print(
        "Block 8: skipping rolling windows larger than the available clean daily returns: "
        f"{skipped_rolling_mean_windows}. Available return rows: {available_return_count:,}."
    )

if rolling_mean_window not in rolling_mean_windows:
    rolling_mean_window = (
        preferred_rolling_mean_window
        if preferred_rolling_mean_window in rolling_mean_windows
        else max(rolling_mean_windows)
    )


def rolling_mean_metric_frames(daily_returns, window):
    window = int(window)
    returns = pd.Series(daily_returns).dropna().sort_index()
    arithmetic_mean = returns.rolling(
        window,
        min_periods=window,
    ).mean()
    geometric_mean = np.expm1(
        np.log1p(returns).rolling(
            window,
            min_periods=window,
        ).mean()
    )
    compounding_efficiency = compounding_efficiency_from_means(arithmetic_mean, geometric_mean)
    volatility_drag = volatility_drag_from_means(arithmetic_mean, geometric_mean)
    arithmetic_label = rolling_arithmetic_label(window)
    geometric_label = rolling_geometric_label(window)

    values = pd.concat(
        {
            arithmetic_label: arithmetic_mean,
            geometric_label: geometric_mean,
            compounding_efficiency_label: compounding_efficiency,
            volatility_drag_label: volatility_drag,
        },
        axis=1,
    ).dropna(how="all")
    mad_scores = pd.concat(
        {
            arithmetic_label: rolling_metric_mad_score(arithmetic_mean),
            geometric_label: rolling_metric_mad_score(geometric_mean),
            compounding_efficiency_label: rolling_metric_mad_score(compounding_efficiency),
            volatility_drag_label: rolling_metric_mad_score(volatility_drag),
        },
        axis=1,
    ).dropna(how="all")
    return values, mad_scores

asset_rolling_mean_detail = {}
for window in rolling_mean_windows:
    values, mad_scores = rolling_mean_metric_frames(rolling_mean_returns, window)
    if not values.empty:
        asset_rolling_mean_detail[window] = {
            "values": values,
            "mad_scores": mad_scores,
        }
if not asset_rolling_mean_detail:
    raise ValueError(
        "Block 8 could not build rolling mean data after cleaning the return series. "
        f"Clean return rows: {available_return_count:,}; requested windows: {requested_rolling_mean_windows}; "
        f"usable windows: {rolling_mean_windows}. "
        "If this is zero or unexpectedly small, rerun Blocks 4, 5, and 7 and check that data retrieval returned a non-empty Close series."
    )
if rolling_mean_window not in asset_rolling_mean_detail:
    rolling_mean_window = next(iter(asset_rolling_mean_detail))

rolling_mean_comparison = asset_rolling_mean_detail[rolling_mean_window]["values"]
rolling_mean_mad_comparison = asset_rolling_mean_detail[rolling_mean_window]["mad_scores"]

benchmark_rolling_mean_detail = {window: {} for window in asset_rolling_mean_detail}
for benchmark_symbol, benchmark_frame in benchmark_data.items():
    if isinstance(benchmark_frame, pd.DataFrame):
        if "Close" not in benchmark_frame:
            continue
        benchmark_close = benchmark_frame["Close"]
    else:
        benchmark_close = pd.Series(benchmark_frame)

    benchmark_returns = benchmark_close.pct_change(fill_method=None)
    for window in asset_rolling_mean_detail:
        benchmark_values, benchmark_mad_scores = rolling_mean_metric_frames(benchmark_returns, window)
        if benchmark_values.empty:
            continue
        benchmark_rolling_mean_detail[window][benchmark_symbol] = {
            "values": benchmark_values,
            "mad_scores": benchmark_mad_scores,
        }

benchmark_overlay_order = []
for window_payload in benchmark_rolling_mean_detail.values():
    for benchmark_symbol in window_payload:
        if benchmark_symbol not in benchmark_overlay_order:
            benchmark_overlay_order.append(benchmark_symbol)
benchmark_line_dashes = ["dot", "dash", "longdash", "dashdot"]
benchmark_line_dash_map = {
    symbol: benchmark_line_dashes[index % len(benchmark_line_dashes)]
    for index, symbol in enumerate(benchmark_overlay_order)
}

rolling_mean_subplot_count = 2

fig_rolling_mean_comparison = make_subplots(
    rows=rolling_mean_subplot_count,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.50, 0.50],
    subplot_titles=(
        f"{compounding_efficiency_label} MAD Score",
        f"{volatility_drag_label} MAD Score",
    ),
)

def block8_plot_title(window):
    return f"{ticker_str}: {int(window)}-Day Compounding Efficiency and Volatility Drag MAD Scores vs Benchmarks"

def metric_trace_specs(window):
    return [
        (compounding_efficiency_label, 1, "#A3E635", "Efficiency", ".4f", True),
        (volatility_drag_label, 2, "#E879F9", "Volatility Drag", ".2%", True),
    ]

def add_metric_trace(
    name_prefix,
    values,
    mad_scores,
    metric_label,
    row,
    color,
    raw_label,
    raw_format,
    line_dash="solid",
    line_width=2.0,
    opacity=1.0,
    visible=True,
):
    series = mad_scores.get(metric_label, pd.Series(dtype=float)).dropna()
    if series.empty:
        return

    raw_value_template = "%{customdata[0]:" + raw_format + "}"
    hovertemplate = (
        "%{x|%Y-%m-%d}<br>"
        "MAD Score: %{y:.2f}<br>"
        f"{raw_label}: " + raw_value_template + "<extra></extra>"
    )
    fig_rolling_mean_comparison.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            mode="lines",
            name=f"{name_prefix} {metric_label} MAD Score",
            line=dict(color=color, width=line_width, dash=line_dash),
            opacity=opacity,
            visible=visible,
            customdata=metric_detail_array(
                values.get(metric_label, pd.Series(dtype=float)),
                mad_scores.get(metric_label, pd.Series(dtype=float)),
                series.index,
            ),
            hovertemplate=hovertemplate,
        ),
        row=row,
        col=1,
    )

window_trace_indices = {}
for window, asset_payload in asset_rolling_mean_detail.items():
    window_visible = window == rolling_mean_window
    trace_start = len(fig_rolling_mean_comparison.data)
    for metric_label, row, color, raw_label, raw_format, _ in metric_trace_specs(window):
        add_metric_trace(
            name_prefix=ticker_str,
            values=asset_payload["values"],
            mad_scores=asset_payload["mad_scores"],
            metric_label=metric_label,
            row=row,
            color=color,
            raw_label=raw_label,
            raw_format=raw_format,
            visible=window_visible,
        )

    for benchmark_symbol, benchmark_payload in benchmark_rolling_mean_detail.get(window, {}).items():
        for metric_label, row, color, raw_label, raw_format, _ in metric_trace_specs(window):
            add_metric_trace(
                name_prefix=benchmark_symbol,
                values=benchmark_payload["values"],
                mad_scores=benchmark_payload["mad_scores"],
                metric_label=metric_label,
                row=row,
                color=color,
                raw_label=raw_label,
                raw_format=raw_format,
                line_dash=benchmark_line_dash_map.get(benchmark_symbol, "dot"),
                line_width=1.4,
                opacity=0.82,
                visible=window_visible,
            )
    window_trace_indices[window] = list(range(trace_start, len(fig_rolling_mean_comparison.data)))

total_trace_count = len(fig_rolling_mean_comparison.data)
base_plot_title = block8_plot_title(rolling_mean_window)

def rolling_window_visibility(window):
    active_indices = set(window_trace_indices.get(window, []))
    return [trace_index in active_indices for trace_index in range(total_trace_count)]

rolling_window_order = list(asset_rolling_mean_detail)
rolling_window_dropdown_buttons = [
    dict(
        label=f"{int(window)} Days",
        method="update",
        args=[
            {"visible": rolling_window_visibility(window)},
            {"title": header_title(block8_plot_title(window))},
        ],
    )
    for window in rolling_window_order
]
rolling_window_dropdown_active = rolling_window_order.index(rolling_mean_window)

plot_x_indexes = [payload["mad_scores"].index for payload in asset_rolling_mean_detail.values()]
for window_payload in benchmark_rolling_mean_detail.values():
    plot_x_indexes.extend(
        payload["mad_scores"].index
        for payload in window_payload.values()
        if not payload["mad_scores"].empty
    )
plot_x_indexes = [index for index in plot_x_indexes if len(index) > 0]
plot_start = min((index.min() for index in plot_x_indexes), default=None)
plot_end = max((index.max() for index in plot_x_indexes), default=None)

block8_updatemenus = [
    block8_dropdown_menu(
        rolling_window_dropdown_buttons,
        x=0.0,
        active=rolling_window_dropdown_active,
    )
]
if plot_start is not None and plot_end is not None:
    default_start = max(plot_start, plot_end - pd.DateOffset(years=10))
    for subplot_row in range(1, rolling_mean_subplot_count + 1):
        fig_rolling_mean_comparison.update_xaxes(range=[default_start, plot_end], row=subplot_row, col=1)
    block8_updatemenus.append(
        block8_dropdown_menu(
            block8_time_range_buttons(plot_start, plot_end, rolling_mean_subplot_count),
            x=0.24,
            active=0,
        )
    )

for score_row in (1, 2):
    is_efficiency_row = score_row == 1
    lower_zone_color = "rgba(34, 197, 94, 0.16)" if is_efficiency_row else "rgba(239, 68, 68, 0.16)"
    upper_zone_color = "rgba(239, 68, 68, 0.16)" if is_efficiency_row else "rgba(34, 197, 94, 0.16)"
    neutral_lower_bound = -1 if is_efficiency_row else -0.5
    neutral_upper_bound = 0.5 if is_efficiency_row else 1
    lower_zone_bounds = (-2, -1) if is_efficiency_row else (-1, -0.5)
    upper_zone_bounds = (0.5, 1) if is_efficiency_row else (1, 2)
    positive_reference_levels = (0.5, 1) if is_efficiency_row else (1, 2)
    negative_reference_levels = (1, 2) if is_efficiency_row else (0.5, 1)
    fig_rolling_mean_comparison.add_hrect(
        y0=neutral_lower_bound,
        y1=neutral_upper_bound,
        fillcolor="rgba(148, 163, 184, 0.12)",
        line_width=0,
        layer="below",
        row=score_row,
        col=1,
    )
    fig_rolling_mean_comparison.add_hrect(
        y0=lower_zone_bounds[0],
        y1=lower_zone_bounds[1],
        fillcolor=lower_zone_color,
        line_width=0,
        layer="below",
        row=score_row,
        col=1,
    )
    fig_rolling_mean_comparison.add_hrect(
        y0=upper_zone_bounds[0],
        y1=upper_zone_bounds[1],
        fillcolor=upper_zone_color,
        line_width=0,
        layer="below",
        row=score_row,
        col=1,
    )
    fig_rolling_mean_comparison.add_hline(
        y=0,
        line_dash="solid",
        line_color="rgba(226, 232, 240, 0.70)",
        row=score_row,
        col=1,
    )
    for sigma_level in positive_reference_levels:
        fig_rolling_mean_comparison.add_hline(
            y=sigma_level,
            line_dash="dash",
            line_color="rgba(148, 163, 184, 0.55)",
            row=score_row,
            col=1,
        )
    for sigma_level in negative_reference_levels:
        fig_rolling_mean_comparison.add_hline(
            y=-sigma_level,
            line_dash="dash",
            line_color="rgba(148, 163, 184, 0.55)",
            row=score_row,
            col=1,
        )

score_zone_annotations = [
    (1, "Efficient Compounding", 0.75, "rgba(255, 235, 235, 0.96)"),
    (1, "Poor Compounding", -1.5, "rgba(235, 255, 235, 0.96)"),
    (2, "High Drag", 1.5, "rgba(235, 255, 235, 0.96)"),
    (2, "Low Drag", -0.75, "rgba(255, 235, 235, 0.96)"),
]

for zone_row, zone_label, zone_y, zone_color in score_zone_annotations:
    fig_rolling_mean_comparison.add_annotation(
        x=0.5,
        y=zone_y,
        xref="x domain",
        yref="y",
        text=zone_label,
        showarrow=False,
        xanchor="center",
        yanchor="middle",
        font=dict(color=zone_color, size=14),
        row=zone_row,
        col=1,
    )

add_formula_annotation(
    fig_rolling_mean_comparison,
    mad_score_annotation_text(
        compounding_efficiency_label,
        "CE",
        r"\frac{1 + g_t}{1 + a_t}",
        "compounding_efficiency_from_means(arithmetic_mean, geometric_mean)",
    ),
    y_position=0.86,
)
add_formula_annotation(
    fig_rolling_mean_comparison,
    mad_score_annotation_text(
        volatility_drag_label,
        "VD",
        r"a_t - g_t",
        "volatility_drag_from_means(arithmetic_mean, geometric_mean)",
    ),
    y_position=0.38,
)

fig_rolling_mean_comparison.update_layout(
    title=header_title(base_plot_title),
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=header_margin(top=165),
    updatemenus=block8_updatemenus,
    height=1125,
)
fig_rolling_mean_comparison.update_yaxes(title_text="MAD Score", tickformat=".2f", range=[-6, 2], row=1, col=1)
fig_rolling_mean_comparison.update_yaxes(title_text="MAD Score", tickformat=".2f", range=[-2, 6], row=2, col=1)
fig_rolling_mean_comparison.update_xaxes(title_text="Date", row=2, col=1)
fig_rolling_mean_comparison = finalize_dark_figure(fig_rolling_mean_comparison)
# Block 8 rows are rendered inside Block 18.


In [ ]:
# Block 10: stack candlestick, drawdown comparison, and rolling recovery time
# Change selected_time_frames in Block 4 to a list like [21, 50, 200], then rerun this cell.

drawdown_recovery_by_window = {}

for window in selected_time_frames:
    
    rolling_peak = compute.rolling(
        asset_history['Close'],
        metric=pd.Series.max,
        window=window,
        min_periods=1,
    )
    
    underwater_series = asset_history['Close'].div(rolling_peak).sub(1.0).dropna()
    
    drawdown_series = compute.rolling(
        asset_history['Close'],
        metric=metric.textbook_window_drawdown,
        window=window,
        dropna=False,
    ).dropna()
    
    recovery_series = compute.rolling(
        asset_history['Close'],
        metric=metric.window_recovery_time,
        window=window,
        dropna=False,
    ).dropna()

    drawdown_recovery_by_window[window] = {
        'underwater': underwater_series,
        'max_drawdown': drawdown_series,
        'recovery_time': recovery_series,
    }

fig_block10_drawdown_recovery = plot_candlestick_drawdown_recovery_view(
    price_frame=asset_history,
    drawdown_recovery_by_window=drawdown_recovery_by_window,
    ticker_label=ticker_str,
    candlestick_period=period,
    default_timeframe_label='10 Years',
)
fig = fig_block10_drawdown_recovery


In [ ]:
# Block 13: visualize monthly and quarterly seasonality patterns

ticker_quarterly_data = series_transforms.resample(asset_history, frequency="quarterly")
ticker_quarterly_returns = ticker_quarterly_data["Close"].pct_change(fill_method=None).dropna()

fig_ticker_seasonality_stack = plot_seasonality_stack_view(
    monthly_returns=ticker_monthly_returns,
    quarterly_returns=ticker_quarterly_returns,
    ticker_label=ticker_str,
    as_of=asset_history.index.max(),
)


In [ ]:
# Block 14: compute Sharpe/Sortino ratios and spreads

from Quantapp.analytics.series_utils import calculate_zscore

asset_close = asset_history['Close'].dropna().sort_index()
annualization_factor = 252
risk_time_frame_map = {str(term): int(window) for term, window in time_frame_map.items()}
selected_windows = sorted(dict.fromkeys(int(window) for window in selected_time_frames))
selected_time_frame_map = {}

for window in selected_windows:
    term_key = next(
        (term for term, mapped_window in risk_time_frame_map.items() if mapped_window == window),
        f"selected_{window}",
    )
    selected_time_frame_map[term_key] = window

risk_time_frame_map.update(selected_time_frame_map)

def rolling_ratio_series(close, window, ratio_type):
    ratio_frame = risk_adjusted_returns(
        close.dropna().sort_index(),
        windows=[window],
        ratio_type=ratio_type,
        risk_free_rate=risk_free_daily_rate,
    )
    return ratio_frame.iloc[:, 0]

def rolling_risk_components(close, window):
    close = close.dropna().sort_index()
    if isinstance(risk_free_daily_rate, pd.Series):
        periodic_risk_free_rate = risk_free_daily_rate.astype(float).sort_index().reindex(close.index).ffill()
    else:
        periodic_risk_free_rate = (1.0 + float(risk_free_daily_rate)) ** (1.0 / annualization_factor) - 1.0
    excess_returns = close.pct_change() - periodic_risk_free_rate
    rolling_mean = excess_returns.rolling(window).mean()
    rolling_std = excess_returns.rolling(window).std()
    sharpe_ratio = np.sqrt(annualization_factor) * rolling_mean / rolling_std
    return {
        "annualized_excess_return": annualization_factor * rolling_mean,
        "annualized_volatility": np.sqrt(annualization_factor) * rolling_std,
        "sharpe_ratio": sharpe_ratio.where(rolling_std > 0).replace([np.inf, -np.inf], np.nan),
    }

def zscore_or_empty(series):
    clean = pd.Series(series).dropna()
    return calculate_zscore(clean).dropna() if not clean.empty else pd.Series(dtype=float)

asset_sharpe_map = {}
asset_sortino_map = {}
asset_component_map = {}

for term, window in risk_time_frame_map.items():
    asset_sharpe_map[term] = rolling_ratio_series(asset_close, window, "sharpe")
    asset_sortino_map[term] = rolling_ratio_series(asset_close, window, "sortino")
    asset_component_map[term] = rolling_risk_components(asset_close, window)

asset_sharpe_sortino_spread_map = {
    term: asset_sortino_map[term] - asset_sharpe_map[term]
    for term in risk_time_frame_map
}

benchmark_metrics = {}
for symbol, benchmark_frame in benchmark_data.items():
    benchmark_close = benchmark_frame["Close"] if isinstance(benchmark_frame, pd.DataFrame) else benchmark_frame
    benchmark_close = benchmark_close.dropna().sort_index()
    benchmark_metrics[symbol] = {}

    for term, window in risk_time_frame_map.items():
        benchmark_components = rolling_risk_components(benchmark_close, window)
        benchmark_sharpe = benchmark_components["sharpe_ratio"]
        benchmark_metrics[symbol][term] = {
            "spread": benchmark_close.pct_change(window) - asset_close.pct_change(window),
            "annualized_excess_return": benchmark_components["annualized_excess_return"],
            "annualized_volatility": benchmark_components["annualized_volatility"],
            "sharpe_ratio": benchmark_sharpe,
            "sharpe_spread": benchmark_sharpe - asset_sharpe_map[term],
        }

benchmark_order = list(benchmark_metrics)
default_benchmark = benchmark_order[0] if benchmark_order else None
spread_plot_data = {
    term: {symbol: benchmark_metrics[symbol][term]["sharpe_spread"] for symbol in benchmark_order}
    for term in risk_time_frame_map
}

term_config_map = {}
for term, window in risk_time_frame_map.items():
    label = f"{window}-day"
    sharpe = asset_sharpe_map[term]
    sortino = asset_sortino_map[term]
    spread = asset_sharpe_sortino_spread_map[term]
    term_config_map[label] = {
        "sharpe": sharpe,
        "sortino": sortino,
        "spread": spread,
        "sharpe_zscore": zscore_or_empty(sharpe),
        "sortino_zscore": zscore_or_empty(sortino),
        "spread_zscore": zscore_or_empty(spread),
        "time_frame": window,
        "term_key": term,
    }

selected_term_config_map = {
    f"{window}-day": term_config_map[f"{window}-day"]
    for window in selected_time_frame_map.values()
    if f"{window}-day" in term_config_map
}


In [ ]:
# Block 15: plot rolling correlation of the asset versus benchmarks

asset_daily_returns = asset_history['Close'].pct_change(fill_method=None)
correlation_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
correlation_benchmark_order = benchmark_order if benchmark_order else list(benchmark_data.keys())
rolling_correlation_map = {}

for term in correlation_term_order:
    window = int(time_frame_map[term])
    term_series_map = {}

    for symbol in correlation_benchmark_order:
        benchmark_frame = benchmark_data.get(symbol)
        if benchmark_frame is None or 'Close' not in benchmark_frame:
            continue

        benchmark_daily_returns = benchmark_frame['Close'].pct_change(fill_method=None)
        aligned_returns = pd.concat(
            [
                asset_daily_returns.rename('asset'),
                benchmark_daily_returns.rename(symbol),
            ],
            axis=1,
        ).dropna()
        if aligned_returns.empty:
            continue

        rolling_correlation_series = aligned_returns['asset'].rolling(window).corr(aligned_returns[symbol]).dropna()
        if rolling_correlation_series.empty:
            continue

        term_series_map[symbol] = rolling_correlation_series

    if term_series_map:
        rolling_correlation_map[term] = term_series_map

rolling_correlation_fig = plot_rolling_correlation_view(
    rolling_correlation_map=rolling_correlation_map,
    time_frame_map=time_frame_map,
    term_order=correlation_term_order,
    benchmark_order=correlation_benchmark_order,
    ticker_label=ticker_str,
)


In [ ]:
# Block 16: Upside Efficiency
# Change selected_time_frames in Block 4 to a list like [21, 50, 200], then rerun the notebook.

fig_block16_upside_efficiency = plot_sharpe_sortino_comparison(
    term_config_map=selected_term_config_map,
    ticker_label=ticker_str,
)
fig = fig_block16_upside_efficiency


In [ ]:
# Block 18: combine risk-adjusted return and benchmark plots
# Requires the current kernel session to have fresh outputs from Blocks 2, 7, and 14.
# Enter the rolling window below, then press Update to rebuild the decomposition.

import copy
import socket

from dash import Dash, Input, Output, Patch, State, ctx, dcc, html, no_update
from plotly import graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import UnivariateSpline
from scipy.signal import find_peaks
from Quantapp.visualization.views.single_asset_profile.pricing.momentum_efficiency._shared import (
    finalize_dark_figure,
    header_margin,
    header_title,
    trace_datetime_bounds,
)

def benchmark_zscore_for_plot(series):
    clean = pd.Series(series).dropna().sort_index()
    if clean.empty:
        return pd.Series(dtype=float)
    zscore_series = calculate_zscore(clean)
    if zscore_series.isna().all():
        return pd.Series(0.0, index=clean.index)
    return zscore_series.dropna()

def _block18_close_series(frame_or_series, label):
    if isinstance(frame_or_series, pd.DataFrame):
        if "Close" not in frame_or_series:
            raise ValueError(f"{label} is missing a Close column.")
        close = frame_or_series["Close"]
    else:
        close = pd.Series(frame_or_series)
    close = pd.to_numeric(close, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().sort_index()
    if close.empty:
        raise ValueError(f"{label} has no usable Close values.")
    return close

def _block18_validate_window(value):
    try:
        window = int(value)
    except (TypeError, ValueError):
        raise ValueError("Enter a whole-number rolling window, for example 21, 50, or 200.")
    if window < 2:
        raise ValueError("The Block 18 rolling window must be at least 2 days.")
    return window

def build_block18_decomposition_figure(window, benchmark_symbols=None):
    if not benchmark_order:
        raise ValueError("No benchmark data available for benchmark comparison plots.")

    window = _block18_validate_window(window)
    requested_benchmark_symbols = [
        symbol for symbol in (benchmark_symbols or []) if symbol in benchmark_order
    ]
    if not requested_benchmark_symbols:
        requested_benchmark_symbols = [default_benchmark or benchmark_order[0]]
    benchmark_symbol = requested_benchmark_symbols[0]

    asset_close_for_window = _block18_close_series(asset_history, ticker_str)
    minimum_rows = window + 2
    if len(asset_close_for_window) < minimum_rows:
        raise ValueError(
            f"{ticker_str} only has {len(asset_close_for_window):,} price rows; "
            f"Block 18 needs at least {minimum_rows:,} rows for a {window}-day window."
        )
    term_key = "custom"
    block18_time_frame_map = {term_key: window}
    asset_sharpe = rolling_ratio_series(asset_close_for_window, window, "sharpe")
    asset_components = rolling_risk_components(asset_close_for_window, window)
    benchmark_detail_payloads = {}
    skipped_benchmark_details = {}
    for candidate_symbol in requested_benchmark_symbols:
        try:
            candidate_close = _block18_close_series(benchmark_data.get(candidate_symbol), candidate_symbol)
        except ValueError as error:
            skipped_benchmark_details[candidate_symbol] = str(error)
            continue
        if len(candidate_close) < minimum_rows:
            skipped_benchmark_details[candidate_symbol] = (
                f"only {len(candidate_close):,} rows; needs {minimum_rows:,}"
            )
            continue

        candidate_components = rolling_risk_components(candidate_close, window)
        candidate_sharpe = candidate_components["sharpe_ratio"]
        benchmark_detail_payloads[candidate_symbol] = {
            "asset": benchmark_zscore_for_plot(asset_sharpe),
            "benchmark": benchmark_zscore_for_plot(candidate_sharpe),
            "asset_sharpe": asset_sharpe.dropna(),
            "benchmark_sharpe": candidate_sharpe.dropna(),
            "asset_excess_return": asset_components.get("annualized_excess_return", pd.Series(dtype=float)).dropna(),
            "benchmark_excess_return": candidate_components.get("annualized_excess_return", pd.Series(dtype=float)).dropna(),
            "asset_volatility": asset_components.get("annualized_volatility", pd.Series(dtype=float)).dropna(),
            "benchmark_volatility": candidate_components.get("annualized_volatility", pd.Series(dtype=float)).dropna(),
            "sharpe_spread": benchmark_zscore_for_plot(candidate_sharpe - asset_sharpe),
            "relative_spread": benchmark_zscore_for_plot(
                candidate_close.pct_change(window) - asset_close_for_window.pct_change(window)
            ),
        }

    plotted_benchmark_symbols = [
        symbol for symbol in requested_benchmark_symbols if symbol in benchmark_detail_payloads
    ]
    if not plotted_benchmark_symbols:
        raise ValueError("No benchmark has enough usable history for the selected rolling window.")
    if benchmark_symbol not in benchmark_detail_payloads:
        benchmark_symbol = plotted_benchmark_symbols[0]
    if skipped_benchmark_details:
        print(
            "Block 18 skipped benchmark overlays: "
            + "; ".join(f"{symbol}: {reason}" for symbol, reason in skipped_benchmark_details.items())
        )

    detail_zscore_map = {
        benchmark_symbol: {term_key: benchmark_detail_payloads[benchmark_symbol]}
    }

    detail_fig = plot_benchmark_zscore_detail(
        detail_zscore_map=detail_zscore_map,
        benchmark_order=[benchmark_symbol],
        time_frame_map=block18_time_frame_map,
        ticker_label=ticker_str,
        default_benchmark=benchmark_symbol,
        default_term=term_key,
    )

    overlay_colors = ["#f97316", "#a78bfa", "#14b8a6", "#facc15", "#fb7185"]
    overlay_dashes = ["dash", "dot", "longdash", "dashdot"]
    for overlay_index, overlay_symbol in enumerate(plotted_benchmark_symbols):
        if overlay_symbol == benchmark_symbol:
            continue
        overlay_payload = benchmark_detail_payloads[overlay_symbol]
        overlay_color = overlay_colors[overlay_index % len(overlay_colors)]
        overlay_dash = overlay_dashes[overlay_index % len(overlay_dashes)]
        overlay_zscore = overlay_payload["benchmark"].dropna()
        if not overlay_zscore.empty:
            detail_fig.add_trace(
                go.Scatter(
                    x=overlay_zscore.index,
                    y=overlay_zscore,
                    mode="lines",
                    name=f"{overlay_symbol} {window}-Day Sharpe Z-Score",
                    legendgroup=f"{overlay_symbol}-{term_key}",
                    line=dict(color=overlay_color, dash=overlay_dash, width=2.2),
                    showlegend=True,
                    hovertemplate=(
                        f"{overlay_symbol}<br>%{{x|%Y-%m-%d}}<br>Sharpe Z-Score: %{{y:.2f}}<extra></extra>"
                    ),
                ),
                row=1,
                col=1,
            )
        overlay_spread = overlay_payload["sharpe_spread"].dropna()
        if not overlay_spread.empty:
            detail_fig.add_trace(
                go.Scatter(
                    x=overlay_spread.index,
                    y=overlay_spread,
                    mode="lines",
                    name=f"{overlay_symbol} - {ticker_str} {window}-Day Sharpe Spread Z-Score",
                    legendgroup=f"{overlay_symbol}-{term_key}",
                    line=dict(color=overlay_color, dash=overlay_dash, width=2.0),
                    showlegend=False,
                    hovertemplate=(
                        f"{overlay_symbol} - {ticker_str}<br>%{{x|%Y-%m-%d}}<br>Spread Z-Score: %{{y:.2f}}<extra></extra>"
                    ),
                ),
                row=2,
                col=1,
            )
    detail_fig.update_layout(updatemenus=[])
    rolling_mean_rows_fig = build_block18_rolling_mean_rows_figure(
        window, plotted_benchmark_symbols
    )
    return _block18_combine_detail_and_mad_rows(
        detail_fig, rolling_mean_rows_fig, window, plotted_benchmark_symbols
    )

def _block18_rolling_metric_mad_score(series):
    clean = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna().sort_index()
    if clean.empty:
        return pd.Series(dtype=float)
    median = clean.median()
    mad = (clean - median).abs().median()
    if mad == 0 or pd.isna(mad):
        return pd.Series(0.0, index=clean.index)
    return ((clean - median) / (1.4826 * mad)).dropna()

def _block18_metric_detail_array(value_series, mad_score_series, index):
    detail_frame = pd.concat(
        {
            "value": pd.Series(value_series),
            "mad_score": pd.Series(mad_score_series),
        },
        axis=1,
    ).reindex(index)
    return detail_frame[["value", "mad_score"]].to_numpy()

def _block18_rolling_mean_metric_frames(close_or_returns, window, *, values_are_returns=False):
    window = _block18_validate_window(window)
    series = pd.to_numeric(pd.Series(close_or_returns), errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().sort_index()
    returns = series if values_are_returns else series.pct_change(fill_method=None)
    returns = pd.Series(returns).replace([np.inf, -np.inf], np.nan).dropna().sort_index()
    if len(returns) < window:
        raise ValueError(f"Need at least {window:,} clean return rows; only {len(returns):,} are available.")
    arithmetic_mean = returns.rolling(window, min_periods=window).mean()
    geometric_mean = np.expm1(np.log1p(returns).rolling(window, min_periods=window).mean())
    compounding_efficiency = (1.0 + geometric_mean).div(1.0 + arithmetic_mean).replace([np.inf, -np.inf], np.nan)
    volatility_drag = arithmetic_mean - geometric_mean
    values = pd.concat(
        {
            "Compounding Efficiency": compounding_efficiency,
            "Volatility Drag": volatility_drag,
        },
        axis=1,
    ).dropna(how="all")
    mad_scores = pd.concat(
        {
            "Compounding Efficiency": _block18_rolling_metric_mad_score(compounding_efficiency),
            "Volatility Drag": _block18_rolling_metric_mad_score(volatility_drag),
        },
        axis=1,
    ).dropna(how="all")
    return values, mad_scores

def _block18_add_mad_trace(fig, name_prefix, values, mad_scores, metric_label, row, color, raw_label, raw_format, *, line_dash="solid", line_width=2.0, opacity=1.0):
    series = mad_scores.get(metric_label, pd.Series(dtype=float)).dropna()
    if series.empty:
        return
    raw_value_template = "%{customdata[0]:" + raw_format + "}"
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series,
            mode="lines",
            name=f"{name_prefix} {metric_label} MAD Score",
            line=dict(color=color, width=line_width, dash=line_dash),
            opacity=opacity,
            customdata=_block18_metric_detail_array(
                values.get(metric_label, pd.Series(dtype=float)),
                mad_scores.get(metric_label, pd.Series(dtype=float)),
                series.index,
            ),
            hovertemplate=(
                "%{x|%Y-%m-%d}<br>"
                "MAD Score: %{y:.2f}<br>"
                f"{raw_label}: " + raw_value_template + "<extra></extra>"
            ),
        ),
        row=row,
        col=1,
    )

def _block18_add_mad_score_guides(fig):
    for score_row in (1, 2):
        is_efficiency_row = score_row == 1
        lower_zone_color = "rgba(34, 197, 94, 0.16)" if is_efficiency_row else "rgba(239, 68, 68, 0.16)"
        upper_zone_color = "rgba(239, 68, 68, 0.16)" if is_efficiency_row else "rgba(34, 197, 94, 0.16)"
        neutral_lower_bound = -1 if is_efficiency_row else -0.5
        neutral_upper_bound = 0.5 if is_efficiency_row else 1
        lower_zone_bounds = (-2, -1) if is_efficiency_row else (-1, -0.5)
        upper_zone_bounds = (0.5, 1) if is_efficiency_row else (1, 2)
        positive_reference_levels = (0.5, 1) if is_efficiency_row else (1, 2)
        negative_reference_levels = (1, 2) if is_efficiency_row else (0.5, 1)
        fig.add_hrect(y0=neutral_lower_bound, y1=neutral_upper_bound, fillcolor="rgba(148, 163, 184, 0.12)", line_width=0, layer="below", row=score_row, col=1)
        fig.add_hrect(y0=lower_zone_bounds[0], y1=lower_zone_bounds[1], fillcolor=lower_zone_color, line_width=0, layer="below", row=score_row, col=1)
        fig.add_hrect(y0=upper_zone_bounds[0], y1=upper_zone_bounds[1], fillcolor=upper_zone_color, line_width=0, layer="below", row=score_row, col=1)
        fig.add_hline(y=0, line_dash="solid", line_color="rgba(226, 232, 240, 0.70)", row=score_row, col=1)
        for sigma_level in positive_reference_levels:
            fig.add_hline(y=sigma_level, line_dash="dash", line_color="rgba(148, 163, 184, 0.55)", row=score_row, col=1)
        for sigma_level in negative_reference_levels:
            fig.add_hline(y=-sigma_level, line_dash="dash", line_color="rgba(148, 163, 184, 0.55)", row=score_row, col=1)

    for zone_row, zone_label, zone_y, zone_color in [
        (1, "Efficient Compounding", 0.75, "rgba(255, 235, 235, 0.96)"),
        (1, "Poor Compounding", -1.5, "rgba(235, 255, 235, 0.96)"),
        (2, "High Drag", 1.5, "rgba(235, 255, 235, 0.96)"),
        (2, "Low Drag", -0.75, "rgba(255, 235, 235, 0.96)"),
    ]:
        fig.add_annotation(
            x=0.5,
            y=zone_y,
            xref="x domain",
            yref="y",
            text=zone_label,
            showarrow=False,
            xanchor="center",
            yanchor="middle",
            font=dict(color=zone_color, size=13),
            row=zone_row,
            col=1,
        )

def build_block18_rolling_mean_rows_figure(window, benchmark_symbols=None):
    window = _block18_validate_window(window)
    benchmark_candidates = benchmark_order if benchmark_symbols is None else benchmark_symbols
    benchmark_symbols = [
        symbol for symbol in benchmark_candidates if symbol in benchmark_data
    ]
    if "ticker_daily_returns" in globals():
        asset_values, asset_mad_scores = _block18_rolling_mean_metric_frames(ticker_daily_returns, window, values_are_returns=True)
    else:
        asset_values, asset_mad_scores = _block18_rolling_mean_metric_frames(_block18_close_series(asset_history, ticker_str), window)

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.50, 0.50],
        subplot_titles=("Compounding Efficiency MAD Score", "Volatility Drag MAD Score"),
    )
    metric_specs = [
        ("Compounding Efficiency", 1, "#A3E635", "Efficiency", ".4f"),
        ("Volatility Drag", 2, "#E879F9", "Volatility Drag", ".2%"),
    ]
    for metric_label, row, color, raw_label, raw_format in metric_specs:
        _block18_add_mad_trace(fig, ticker_str, asset_values, asset_mad_scores, metric_label, row, color, raw_label, raw_format)

    benchmark_line_dashes = ["dot", "dash", "longdash", "dashdot"]
    for benchmark_index, benchmark_symbol in enumerate(benchmark_symbols):
        benchmark_frame = benchmark_data[benchmark_symbol]
        if isinstance(benchmark_frame, pd.DataFrame) and "Close" in benchmark_frame:
            benchmark_close = benchmark_frame["Close"]
        else:
            benchmark_close = pd.Series(benchmark_frame)
        benchmark_values, benchmark_mad_scores = _block18_rolling_mean_metric_frames(benchmark_close, window)
        line_dash = benchmark_line_dashes[benchmark_index % len(benchmark_line_dashes)]
        for metric_label, row, color, raw_label, raw_format in metric_specs:
            _block18_add_mad_trace(
                fig,
                benchmark_symbol,
                benchmark_values,
                benchmark_mad_scores,
                metric_label,
                row,
                color,
                raw_label,
                raw_format,
                line_dash=line_dash,
                line_width=1.4,
                opacity=0.82,
            )

    plot_indexes = [trace.x for trace in fig.data if getattr(trace, "x", None) is not None and len(trace.x) > 0]
    plot_start = min((pd.Index(index).min() for index in plot_indexes), default=None)
    plot_end = max((pd.Index(index).max() for index in plot_indexes), default=None)
    if plot_start is not None and plot_end is not None:
        default_start = max(plot_start, plot_end - pd.DateOffset(years=10))
        for subplot_row in (1, 2):
            fig.update_xaxes(range=[default_start, plot_end], row=subplot_row, col=1)

    _block18_add_mad_score_guides(fig)
    fig.update_layout(
        title=header_title(f"{ticker_str}: {window}-Day Compounding Efficiency and Volatility Drag MAD Scores vs Benchmarks"),
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=header_margin(top=120),
        height=760,
    )
    fig.update_yaxes(title_text="MAD Score", tickformat=".2f", range=[-6, 2], row=1, col=1)
    fig.update_yaxes(title_text="MAD Score", tickformat=".2f", range=[-2, 6], row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    return finalize_dark_figure(fig)

def _block18_axis_ref_row(axis_ref):
    if axis_ref is None:
        return 1
    token = str(axis_ref).split()[0]
    if len(token) == 1:
        return 1
    try:
        return int(token[1:])
    except ValueError:
        return 1

def _block18_axis_ref(axis_letter, row, *, domain=False):
    suffix = "" if row == 1 else str(row)
    axis_ref = f"{axis_letter}{suffix}"
    return f"{axis_ref} domain" if domain else axis_ref

def _block18_target_row(source_row, row_offset, row_map):
    if row_map is None:
        return source_row + row_offset
    return row_map.get(source_row)

def _block18_remap_axis_ref(axis_ref, row_offset, row_map=None):
    if not isinstance(axis_ref, str):
        return axis_ref
    token = axis_ref.split()[0]
    if not token or token[0] not in {"x", "y"}:
        return axis_ref
    domain = axis_ref.endswith(" domain")
    target_row = _block18_target_row(_block18_axis_ref_row(token), row_offset, row_map)
    if target_row is None:
        return None
    return _block18_axis_ref(token[0], target_row, domain=domain)

def _block18_copy_subplot_traces(source_fig, target_fig, *, row_offset=0, row_map=None):
    for trace in source_fig.data:
        source_row = _block18_axis_ref_row(getattr(trace, "yaxis", None))
        target_row = _block18_target_row(source_row, row_offset, row_map)
        if target_row is None:
            continue
        target_fig.add_trace(copy.deepcopy(trace), row=target_row, col=1)

def _block18_copy_axis_annotations(source_fig, target_fig, *, row_offset=0, row_map=None):
    for annotation in source_fig.layout.annotations or []:
        annotation_payload = annotation.to_plotly_json()
        if annotation_payload.get("yref") == "paper":
            continue
        annotation_payload["xref"] = _block18_remap_axis_ref(annotation_payload.get("xref"), row_offset, row_map)
        annotation_payload["yref"] = _block18_remap_axis_ref(annotation_payload.get("yref"), row_offset, row_map)
        if annotation_payload["xref"] is None or annotation_payload["yref"] is None:
            continue
        target_fig.add_annotation(**annotation_payload)

def _block18_copy_axis_shapes(source_fig, target_fig, *, row_offset=0, row_map=None):
    for shape in source_fig.layout.shapes or []:
        shape_payload = shape.to_plotly_json()
        shape_payload["xref"] = _block18_remap_axis_ref(shape_payload.get("xref"), row_offset, row_map)
        shape_payload["yref"] = _block18_remap_axis_ref(shape_payload.get("yref"), row_offset, row_map)
        if shape_payload["xref"] is None or shape_payload["yref"] is None:
            continue
        target_fig.add_shape(**shape_payload)

def _block18_combine_detail_and_mad_rows(detail_fig, mad_rows_fig, window, benchmark_symbols):
    benchmark_label = " + ".join(benchmark_symbols)
    combined_fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.045,
        row_heights=[0.36, 0.29, 0.35],
        subplot_titles=(
            "Risk-Adjusted Return Z-Score Comparison",
            "Sharpe Spread Z-Score",
            "Volatility Drag MAD Score",
        ),
    )
    _block18_copy_subplot_traces(detail_fig, combined_fig, row_map={1: 1, 2: 2})
    _block18_copy_axis_annotations(detail_fig, combined_fig, row_map={1: 1, 2: 2})
    _block18_copy_axis_shapes(detail_fig, combined_fig, row_map={1: 1, 2: 2})
    _block18_copy_subplot_traces(mad_rows_fig, combined_fig, row_map={2: 3})
    _block18_copy_axis_annotations(mad_rows_fig, combined_fig, row_map={2: 3})
    _block18_copy_axis_shapes(mad_rows_fig, combined_fig, row_map={2: 3})

    combined_fig.update_yaxes(title_text="Sharpe Z-Score", row=1, col=1)
    combined_fig.update_yaxes(title_text="Spread Z-Score", row=2, col=1)
    combined_fig.update_yaxes(title_text="MAD Score", tickformat=".2f", range=[-2, 6], row=3, col=1)
    combined_fig.update_xaxes(title_text="Date", row=3, col=1)

    detail_start, detail_end = trace_datetime_bounds(combined_fig.data)
    if detail_start is not None and detail_end is not None:
        detail_default_start = max(detail_start, detail_end - pd.DateOffset(years=3))
        combined_fig.update_xaxes(range=[detail_default_start, detail_end])

    combined_fig.update_layout(
        title=header_title(
            f"{ticker_str} vs {benchmark_label} Risk-Adjusted Return and Compounding Diagnostics [{window}-Day]"
        ),
        hovermode="x unified",
        height=2150,
        margin=header_margin(top=150),
        template="plotly_dark",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        updatemenus=[],
    )
    return finalize_dark_figure(combined_fig)

def _block18_error_figure(message, *, height=850):
    error_fig = go.Figure()
    error_fig.add_annotation(
        text=message,
        showarrow=False,
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        font=dict(color="#fca5a5", size=14),
    )
    error_fig.update_layout(
        template="plotly_dark",
        height=height,
        paper_bgcolor="#0b0f14",
        plot_bgcolor="#0b0f14",
        margin=dict(t=48, r=24, b=24, l=24),
    )
    return error_fig

def _block18_available_port(start=8060, stop=8090):
    for port in range(start, stop + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("127.0.0.1", port))
            except OSError:
                continue
            return port
    raise RuntimeError("No open localhost port found for the Block 18 Dash app.")

block18_default_window = int(globals().get("default_window", 200))
block18_default_display_range = "3y"
block18_default_benchmark_selection = []
block18_playback_dates = pd.DatetimeIndex(
    pd.to_datetime(
        momentum_diagnostics_contexts[ticker_str]["sharpe_table"].dropna(how="all").index,
        errors="coerce",
        utc=True,
    )
).dropna().tz_convert(None).normalize().unique().sort_values()
if block18_playback_dates.empty:
    raise ValueError("Block 18 playback needs at least one valid diagnostics date.")
block18_today = pd.Timestamp.today().normalize()
block18_default_playback_position = int(
    block18_playback_dates.searchsorted(block18_today, side="right") - 1
)
block18_default_playback_position = max(
    0, min(block18_default_playback_position, len(block18_playback_dates) - 1)
)
block18_playback_step = 5  # Advance five trading sessions per timer tick.
block18_continuous_playback_interval = 75
block18_playback_speed_options = [
    {"label": "Continuous (no hold)", "value": "continuous"},
    {"label": "Slow", "value": 750},
    {"label": "Normal", "value": 300},
    {"label": "Fast", "value": 120},
]
block18_default_playback_speed = "continuous"
block18_default_playback_interval = block18_continuous_playback_interval

def _block18_animation_options(speed_milliseconds):
    continuous = speed_milliseconds == "continuous"
    if continuous:
        frame_duration = block18_continuous_playback_interval
    else:
        try:
            frame_duration = max(100, int(speed_milliseconds))
        except (TypeError, ValueError):
            continuous = block18_default_playback_speed == "continuous"
            frame_duration = block18_default_playback_interval
    transition_duration = (
        frame_duration if continuous else max(80, frame_duration - 20)
    )
    return {
        "frame": {"duration": frame_duration, "redraw": False},
        "transition": {"duration": transition_duration, "easing": "linear"},
        "mode": "immediate",
        "fromcurrent": True,
    }
block18_playback_mark_positions = sorted(set(
    int(position) for position in np.linspace(0, len(block18_playback_dates) - 1, num=min(8, len(block18_playback_dates)))
))
block18_playback_marks = {
    position: block18_playback_dates[position].strftime("%Y-%m-%d")
    for position in block18_playback_mark_positions
}
block18_base_figure_cache = {}
block18_window_diagnostics_playback_cache = {}
block18_diagnostics_window_min = int(min(window_sizes))
block18_diagnostics_window_max = int(max(window_sizes))
block18_default_diagnostics_window_range = [
    block18_diagnostics_window_min, block18_diagnostics_window_max
]
block18_diagnostics_window_marks = {
    value: str(value)
    for value in sorted(set([
        block18_diagnostics_window_min,
        30, 60, 90, 120, 180, 252, 300, 400, 540, 720,
        block18_diagnostics_window_max,
    ]))
    if block18_diagnostics_window_min <= value <= block18_diagnostics_window_max
}
block18_default_spline_toggle = ["cubic_bspline"]
block18_default_raw_overlay_toggle = ["raw_overlay"]
block18_default_spline_strength = 25
block18_horizon_derivative_scale = 20.0
block18_extrema_min_prominence = 0.12
block18_extrema_prominence_fraction = 0.08
block18_asset_profile_color = "#60a5fa"
block18_benchmark_profile_palette = [
    "#f97316", "#22c55e", "#facc15", "#ef4444", "#ec4899", "#f8fafc"
]

block18_tab_style = {"backgroundColor": "#111827", "color": "#cbd5e1", "borderColor": "#334155"}
block18_selected_tab_style = {"backgroundColor": "#1f2937", "color": "#ffffff", "borderColor": "#60a5fa"}
block18_display_range_options = [
    {"label": "Full", "value": "full"},
    {"label": "10 Years", "value": "10y"},
    {"label": "5 Years", "value": "5y"},
    {"label": "3 Years", "value": "3y"},
    {"label": "2 Years", "value": "2y"},
    {"label": "1 Year", "value": "1y"},
    {"label": "6 Months", "value": "6m"},
    {"label": "3 Months", "value": "3m"},
]
block18_display_range_labels = {option["value"]: option["label"] for option in block18_display_range_options}
block18_display_range_offsets = {
    "full": None,
    "10y": pd.DateOffset(years=10),
    "5y": pd.DateOffset(years=5),
    "3y": pd.DateOffset(years=3),
    "2y": pd.DateOffset(years=2),
    "1y": pd.DateOffset(years=1),
    "6m": pd.DateOffset(months=6),
    "3m": pd.DateOffset(months=3),
}
block18_tab_config = {
    "heatmap": {"label": "Sharpe Heatmap", "height": 1420, "accent": "#14b8a6"},
    "window_diagnostics": {"label": "Functional Horizon Profile", "height": 2750, "accent": "#f59e0b"},
    "risk": {"label": "Risk & Compounding", "height": 2150, "accent": "#60a5fa"},
    "upside": {"label": "Sharpe vs Sortino", "height": 1100, "accent": "#a78bfa"},
    "drawdown": {"label": "Drawdown & Recovery", "height": 1650, "accent": "#f87171"},
    "seasonality": {"label": "Seasonality", "height": 850, "accent": "#34d399"},
    "correlation": {"label": "Rolling Correlation", "height": 850, "accent": "#38bdf8"},
    "volatility_efficiency": {"label": "Volatility & Efficiency", "height": 1200, "accent": "#fb7185"},
}
block18_tab_order = ["heatmap", "window_diagnostics", "risk", "upside", "drawdown", "seasonality", "correlation", "volatility_efficiency"]
block18_date_range_tabs = {"risk", "upside", "drawdown", "correlation", "heatmap", "volatility_efficiency"}
block18_native_axis_status = {
    "seasonality": "Native seasonal axes",
    "window_diagnostics": "Native lookback-horizon/DTE axes",
}

def _block18_graph_height(figure, default_height=1125):
    height = getattr(getattr(figure, "layout", None), "height", None)
    try:
        return f"{int(height)}px"
    except (TypeError, ValueError):
        return f"{int(default_height)}px"

def _block18_tab(value):
    accent = block18_tab_config[value]["accent"]
    return dcc.Tab(
        label=block18_tab_config[value]["label"],
        value=value,
        style={
            **block18_tab_style,
            "borderTop": f"4px solid {accent}",
            "borderBottom": "1px solid #334155",
        },
        selected_style={
            **block18_selected_tab_style,
            "borderTop": f"4px solid {accent}",
            "boxShadow": f"inset 0 3px 0 {accent}",
        },
    )

def _block18_display_range_label(value):
    return block18_display_range_labels.get(value, block18_display_range_labels[block18_default_display_range])

def _block18_strip_figure_controls(figure, *, preserve_benchmark_selector=False):
    stripped = go.Figure(figure)
    if not preserve_benchmark_selector:
        stripped.layout.updatemenus = ()
        return stripped

    benchmark_labels = set(benchmark_order)
    benchmark_menus = []
    for menu in stripped.layout.updatemenus or []:
        button_labels = {str(button.label) for button in menu.buttons or []}
        if button_labels.intersection(benchmark_labels):
            benchmark_menus.append(menu)
    stripped.layout.updatemenus = tuple(benchmark_menus)
    return stripped

def _block18_named_figure(figure_name, label, default_height=1125):
    figure = globals().get(figure_name)
    if figure is None:
        return _block18_error_figure(f"{label} figure is unavailable. Run the source cell first.", height=default_height)
    return figure

def _block18_apply_display_range(
    figure,
    display_range_value,
    *,
    preserve_benchmark_selector=False,
    as_of_date=None,
    show_playhead=False,
):
    figure = _block18_strip_figure_controls(
        figure, preserve_benchmark_selector=preserve_benchmark_selector
    )
    display_range_value = display_range_value if display_range_value in block18_display_range_offsets else block18_default_display_range
    date_start, date_end = trace_datetime_bounds(figure.data)
    if date_start is None or date_end is None:
        return figure

    effective_end = date_end
    if as_of_date is not None:
        requested_end = pd.Timestamp(as_of_date)
        if requested_end.tzinfo is not None:
            requested_end = requested_end.tz_convert(None)
        effective_end = min(date_end, requested_end)
        effective_end = max(date_start, effective_end)

    offset = block18_display_range_offsets[display_range_value]
    range_start = date_start if offset is None else max(date_start, effective_end - offset)
    for axis_name in figure.layout.to_plotly_json():
        if axis_name.startswith("xaxis"):
            figure.layout[axis_name].update(range=[range_start, effective_end])

    if show_playhead:
        figure.add_shape(
            type="line",
            x0=effective_end,
            x1=effective_end,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line={"color": "#facc15", "width": 2, "dash": "dash"},
            name="Playback as-of date",
        )
        figure.add_annotation(
            x=effective_end,
            y=1,
            xref="x",
            yref="paper",
            text=f"As of {effective_end:%Y-%m-%d}",
            showarrow=False,
            xanchor="right",
            yanchor="bottom",
            font={"color": "#facc15", "size": 11},
            bgcolor="rgba(15, 23, 42, 0.80)",
        )
    return figure

def _block18_build_term_config_for_window(window):
    label = f"{window}-day"
    if label in term_config_map:
        return {label: term_config_map[label]}

    sharpe = rolling_ratio_series(asset_close, window, "sharpe")
    sortino = rolling_ratio_series(asset_close, window, "sortino")
    spread = sortino - sharpe
    return {
        label: {
            "sharpe": sharpe,
            "sortino": sortino,
            "spread": spread,
            "sharpe_zscore": zscore_or_empty(sharpe),
            "sortino_zscore": zscore_or_empty(sortino),
            "spread_zscore": zscore_or_empty(spread),
            "time_frame": window,
            "term_key": label,
        }
    }

def _block18_drawdown_recovery_for_window(window):
    rolling_peak = compute.rolling(
        asset_history["Close"],
        metric=pd.Series.max,
        window=window,
        min_periods=1,
    )
    underwater_series = asset_history["Close"].div(rolling_peak).sub(1.0).dropna()
    drawdown_series = compute.rolling(
        asset_history["Close"],
        metric=metric.textbook_window_drawdown,
        window=window,
        dropna=False,
    ).dropna()
    recovery_series = compute.rolling(
        asset_history["Close"],
        metric=metric.window_recovery_time,
        window=window,
        dropna=False,
    ).dropna()
    return {
        window: {
            "underwater": underwater_series,
            "max_drawdown": drawdown_series,
            "recovery_time": recovery_series,
        }
    }

def _block18_rolling_correlation_for_window(window, benchmark_symbols=None):
    asset_returns_for_window = asset_close.pct_change()
    term_label = f"{window}-day"
    term_series_map = {}
    benchmark_candidates = benchmark_order if benchmark_symbols is None else benchmark_symbols
    benchmark_symbols = [
        symbol for symbol in benchmark_candidates if symbol in benchmark_data
    ]
    for symbol in benchmark_symbols:
        benchmark_frame = benchmark_data[symbol]
        benchmark_close = benchmark_frame["Close"] if isinstance(benchmark_frame, pd.DataFrame) else benchmark_frame
        benchmark_returns = pd.Series(benchmark_close).dropna().sort_index().pct_change()
        aligned_returns = pd.concat(
            {"asset": asset_returns_for_window, symbol: benchmark_returns},
            axis=1,
        ).dropna()
        if aligned_returns.empty:
            continue

        rolling_correlation_series = aligned_returns["asset"].rolling(window).corr(aligned_returns[symbol]).dropna()
        if not rolling_correlation_series.empty:
            term_series_map[symbol] = rolling_correlation_series

    return {term_label: term_series_map} if term_series_map else {}

def _block18_benchmark_trace_owner(trace_name):
    trace_name = str(trace_name)
    for symbol in sorted(benchmark_order, key=len, reverse=True):
        if trace_name == symbol or trace_name.startswith(f"{symbol} "):
            return symbol
    return None

def _block18_filter_benchmark_traces(figure, selected_benchmarks):
    filtered = go.Figure(figure)
    filtered.data = tuple(
        trace
        for trace in filtered.data
        if (
            _block18_benchmark_trace_owner(getattr(trace, "name", "")) is None
            or _block18_benchmark_trace_owner(getattr(trace, "name", "")) in selected_benchmarks
        )
    )
    return filtered

def _block18_normalized_shannon_entropy(values, bins=5):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < bins or np.ptp(values) == 0:
        return np.nan
    counts = np.histogram(values, bins=bins)[0]
    probabilities = counts[counts > 0] / counts.sum()
    return float(-(probabilities * np.log(probabilities)).sum() / np.log(bins))

def _block18_hurst_rs(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    candidate_sizes = np.array([8, 16, 32, 64, 128])
    sizes = candidate_sizes[candidate_sizes <= len(values) // 2]
    rs_points = []
    for size in sizes:
        ratios = []
        for start in range(0, len(values) - size + 1, size):
            chunk = values[start:start + size]
            std = chunk.std(ddof=1)
            if std > 0:
                path = np.cumsum(chunk - chunk.mean())
                ratios.append((path.max() - path.min()) / std)
        if ratios:
            rs_points.append((size, np.mean(ratios)))
    if len(rs_points) < 2:
        return np.nan
    x, y = np.asarray(rs_points, dtype=float).T
    return float(np.polyfit(np.log(x), np.log(y), 1)[0])

def _block18_volatility_efficiency_figure():
    vol_window = 21
    efficiency_window = 252
    vol_of_vol_window = 63
    close = pd.to_numeric(asset_history["Close"], errors="coerce").dropna().sort_index()
    returns = np.log(close).diff().dropna()
    if len(returns) < efficiency_window:
        raise ValueError(f"{ticker_str} needs at least {efficiency_window} returns for efficiency metrics.")
    volatility = returns.rolling(vol_window).std() * np.sqrt(252) * 100
    entropy = returns.rolling(efficiency_window).apply(
        _block18_normalized_shannon_entropy, raw=True, kwargs={"bins": 5}
    )
    autocorrelation = returns.rolling(efficiency_window).corr(returns.shift(1))
    hurst = returns.rolling(efficiency_window).apply(_block18_hurst_rs, raw=True)
    vol_of_vol = volatility.rolling(vol_of_vol_window).std()
    metrics = pd.concat({
        "Volatility": volatility,
        "Entropy": entropy,
        "Autocorrelation": autocorrelation,
        "Hurst exponent": hurst,
        "Vol of vol": vol_of_vol,
    }, axis=1)
    titles = [
        f"Volatility ({vol_window}d annualized)",
        f"Normalized Entropy ({efficiency_window}d)",
        f"Lag-1 Autocorrelation ({efficiency_window}d)",
        f"Hurst Exponent ({efficiency_window}d)",
        f"Volatility of Volatility ({vol_of_vol_window}d)",
    ]
    colors = ["#38bdf8", "#a78bfa", "#f59e0b", "#34d399", "#fb7185"]
    figure = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.035, subplot_titles=titles)
    for row, (column, color) in enumerate(zip(metrics.columns, colors), start=1):
        figure.add_trace(go.Scatter(x=metrics.index, y=metrics[column], name=column, mode="lines", line=dict(color=color, width=1.5)), row=row, col=1)
    figure.add_hline(y=1.0, line_dash="dot", line_color="#64748b", row=2, col=1)
    figure.add_hline(y=0.0, line_dash="dot", line_color="#64748b", row=3, col=1)
    figure.add_hline(y=0.5, line_dash="dot", line_color="#64748b", row=4, col=1)
    figure.update_yaxes(title_text="Annualized %", row=1, col=1)
    figure.update_yaxes(title_text="0-1", range=[0, 1.05], row=2, col=1)
    figure.update_yaxes(title_text="Correlation", range=[-1, 1], row=3, col=1)
    figure.update_yaxes(title_text="H", row=4, col=1)
    figure.update_yaxes(title_text="Percentage pts", row=5, col=1)
    figure.update_layout(title=f"{ticker_str} - Volatility & Market Efficiency", template="plotly_dark", height=1200, hovermode="x unified", showlegend=False, margin=dict(t=90, r=30, b=45, l=80))
    return figure

def _block18_build_tab_figure(active_tab, window, display_range_value, selected_benchmarks):
    if active_tab == "volatility_efficiency":
        return _block18_volatility_efficiency_figure()
    if active_tab == "risk":
        return build_block18_decomposition_figure(window, selected_benchmarks)
    if active_tab == "upside":
        term_label = f"{window}-day"
        return plot_sharpe_sortino_comparison(
            term_config_map=_block18_build_term_config_for_window(window),
            ticker_label=ticker_str,
            default_label=term_label,
        )
    if active_tab == "drawdown":
        return plot_candlestick_drawdown_recovery_view(
            price_frame=asset_history,
            drawdown_recovery_by_window=_block18_drawdown_recovery_for_window(window),
            ticker_label=ticker_str,
            candlestick_period=period,
            default_window=window,
            show_window_menu=False,
            default_timeframe_label=_block18_display_range_label(display_range_value),
        )
    if active_tab == "correlation":
        term_label = f"{window}-day"
        return plot_rolling_correlation_view(
            rolling_correlation_map=_block18_rolling_correlation_for_window(window, selected_benchmarks),
            time_frame_map={term_label: window},
            term_order=[term_label],
            benchmark_order=selected_benchmarks,
            ticker_label=ticker_str,
        )
    if active_tab == "seasonality":
        return _block18_named_figure("fig_ticker_seasonality_stack", "Seasonality", default_height=850)
    if active_tab == "heatmap":
        return plot_sharpe_zscore_heatmap_view(
            asset_sharpe_zscore_frame=asset_sharpe_zscore_frame,
            benchmark_sharpe_zscore_frames={
                symbol: benchmark_sharpe_zscore_frames[symbol]
                for symbol in selected_benchmarks if symbol in benchmark_sharpe_zscore_frames
            },
            benchmark_spread_zscore_frames={
                symbol: benchmark_spread_zscore_frames[symbol]
                for symbol in selected_benchmarks if symbol in benchmark_spread_zscore_frames
            },
            benchmark_order=selected_benchmarks,
            default_benchmark=selected_benchmarks[0] if selected_benchmarks else None,
            ticker_label=ticker_str,
        )
    if active_tab == "window_diagnostics":
        _block18_prepare_window_diagnostics_playback()
        return _block18_filter_benchmark_traces(
            _block18_named_figure("fig_momentum_window_diagnostics_grid", "Functional Horizon Profile", default_height=2750),
            selected_benchmarks,
        )
    return _block18_error_figure("Select a valid Momentum & Efficiency view.", height=850)

def _block18_expected_factor_benchmarks():
    if not globals().get("include_factor_peer_index", False):
        return {}
    lookup_symbol = globals().get("factor_peer_index_source_ticker") or ticker_str
    safe_symbol = str(lookup_symbol).replace(".", "_").replace("/", "_")
    expected = {}
    for gics_level, file_suffix in factor_gics_index_suffixes.items():
        cache_path = Path(peer_index_cache_dir) / f"{safe_symbol}_{file_suffix}_index.csv"
        if not cache_path.exists():
            continue
        cached_header = pd.read_csv(cache_path, nrows=1)
        cached_label = cached_header.get("Benchmark Label", pd.Series(dtype="object")).dropna()
        label = str(cached_label.iloc[-1]) if not cached_label.empty else f"{lookup_symbol} {gics_level} Index"
        expected[label] = cache_path
    if expected:
        return expected

    legacy_path = Path(peer_index_cache_dir) / f"{safe_symbol}_peer_index.csv"
    if legacy_path.exists():
        cached_header = pd.read_csv(legacy_path, nrows=1)
        cached_label = cached_header.get("Benchmark Label", pd.Series(dtype="object")).dropna()
        label = str(cached_label.iloc[-1]) if not cached_label.empty else f"{lookup_symbol} Peer Index"
        expected[label] = legacy_path
    return expected

def _block18_validate_peer_state(active_tab, figure=None, required_benchmarks=None):
    expected_benchmarks = _block18_expected_factor_benchmarks()
    if not expected_benchmarks:
        return
    benchmark_candidates = (
        expected_benchmarks if required_benchmarks is None else required_benchmarks
    )
    required_labels = [
        label for label in benchmark_candidates
        if label in expected_benchmarks
    ]
    missing_labels = [label for label in required_labels if label not in benchmark_data]
    if missing_labels:
        raise RuntimeError(
            f"Factor GICS indexes exist but are not loaded in benchmark_data: {missing_labels}. "
            "Restart the kernel and Run All, or rerun Block 5 and every downstream source cell before Block 18."
        )
    if active_tab not in {"heatmap", "window_diagnostics"} or figure is None:
        return
    trace_names = [str(getattr(trace, "name", "")) for trace in figure.data]
    missing_trace_labels = [
        label for label in required_labels
        if not any(label in trace_name for trace_name in trace_names)
    ]
    if missing_trace_labels:
        source_block = "Block 12" if active_tab == "heatmap" else "Block 11"
        raise RuntimeError(
            f"Factor indexes {missing_trace_labels} are loaded but the cached "
            f"{block18_tab_config[active_tab]['label']} figure is stale. "
            f"Rerun {source_block}, then rerun Block 18."
        )

def _block18_normalize_benchmark_selection(selected_benchmarks):
    if isinstance(selected_benchmarks, str):
        selected_benchmarks = [selected_benchmarks]
    selected = [
        symbol for symbol in (selected_benchmarks or []) if symbol in benchmark_order
    ]
    return list(dict.fromkeys(selected))

def _block18_benchmark_selection_label(selected_benchmarks):
    return ", ".join(selected_benchmarks) if selected_benchmarks else "None"

def _block18_normalize_diagnostics_window_range(window_range):
    if not isinstance(window_range, (list, tuple)) or len(window_range) != 2:
        return block18_default_diagnostics_window_range.copy()
    try:
        lower, upper = sorted(int(value) for value in window_range)
    except (TypeError, ValueError):
        return block18_default_diagnostics_window_range.copy()
    lower = max(block18_diagnostics_window_min, lower)
    upper = min(block18_diagnostics_window_max, upper)
    return [lower, max(lower, upper)]

def _block18_apply_diagnostics_window_range(figure, window_range):
    lower, upper = _block18_normalize_diagnostics_window_range(window_range)
    for row in (3, 4, 5, 6, 7, 8):
        figure.update_xaxes(range=[lower, upper], row=row, col=1)
    return figure, [lower, upper]

def _block18_patch_diagnostics_window_range(window_range):
    lower, upper = _block18_normalize_diagnostics_window_range(window_range)
    patched_figure = Patch()
    for axis_number in range(5, 11):
        patched_figure["layout"][f"xaxis{axis_number}"]["range"] = [lower, upper]
    return patched_figure, [lower, upper]

def _block18_spline_is_enabled(toggle_value):
    if isinstance(toggle_value, str):
        toggle_value = [toggle_value]
    return "cubic_bspline" in (toggle_value or [])

def _block18_raw_overlay_is_enabled(toggle_value):
    if isinstance(toggle_value, str):
        toggle_value = [toggle_value]
    return "raw_overlay" in (toggle_value or [])

def _block18_normalize_spline_strength(strength):
    try:
        return max(0.0, min(float(strength), 100.0))
    except (TypeError, ValueError):
        return float(block18_default_spline_strength)

def _block18_is_current_horizon_profile(trace_name):
    trace_name = str(trace_name)
    if trace_name in {"Current Sharpe Z-Score", "Cross-Window Relative Z-Score"}:
        return True
    owner = _block18_benchmark_trace_owner(trace_name)
    return owner is not None and trace_name in {
        f"{owner} Current Sharpe Z-Score",
        f"{owner} Cross-Window Relative Z-Score",
    }

def _block18_spline_trace_name(raw_trace_name):
    return f"{raw_trace_name} — Cubic B-Spline"

def _block18_horizon_derivative_trace_name(raw_trace_name, order, use_spline):
    derivative_label = "First Derivative" if order == 1 else "Second Derivative"
    profile_label = "Cubic B-Spline " if use_spline else ""
    return f"{raw_trace_name} — {profile_label}{derivative_label}"

def _block18_is_horizon_derivative_trace(trace_name):
    trace_name = str(trace_name)
    return trace_name.endswith("First Derivative") or trace_name.endswith("Second Derivative")

def _block18_horizon_extrema_trace_name(raw_trace_name, kind, use_spline):
    profile_label = "Cubic B-Spline " if use_spline else "Raw "
    marker_label = "Peak Markers" if kind == "peak" else "Trough Markers"
    return f"{raw_trace_name} — {profile_label}{marker_label}"

def _block18_is_horizon_extrema_trace(trace_name):
    trace_name = str(trace_name)
    return trace_name.endswith("Peak Markers") or trace_name.endswith("Trough Markers")

def _block18_horizon_transition_trace_name(raw_trace_name, feature, use_spline):
    profile_label = "Cubic B-Spline " if use_spline else "Raw "
    feature_label = (
        "Fastest Movement Markers"
        if feature == "movement" else "Fastest Rate-Change Markers"
    )
    return f"{raw_trace_name} — {profile_label}{feature_label}"

def _block18_horizon_transition_profile_trace_name(raw_trace_name, feature, use_spline):
    profile_label = "Cubic B-Spline " if use_spline else "Raw "
    feature_label = (
        "Fastest Movement Profile Markers"
        if feature == "movement" else "Fastest Rate-Change Profile Markers"
    )
    return f"{raw_trace_name} — {profile_label}{feature_label}"

def _block18_is_horizon_transition_trace(trace_name):
    trace_name = str(trace_name)
    return (
        trace_name.endswith("Fastest Movement Markers")
        or trace_name.endswith("Fastest Rate-Change Markers")
        or trace_name.endswith("Fastest Movement Profile Markers")
        or trace_name.endswith("Fastest Rate-Change Profile Markers")
    )

def _block18_horizon_deceleration_trace_name(raw_trace_name, panel, use_spline):
    profile_label = "Cubic B-Spline " if use_spline else "Raw "
    panel_label = "Profile" if panel == "profile" else "Slope"
    return f"{raw_trace_name} — {profile_label}{panel_label} Deceleration Leg"

def _block18_is_horizon_deceleration_trace(trace_name):
    return str(trace_name).endswith("Deceleration Leg")

def _block18_horizon_derivative_axes(trace_name, order):
    cross_window = str(trace_name).endswith("Cross-Window Relative Z-Score")
    if cross_window:
        return ("x9", "y9") if order == 1 else ("x10", "y10")
    return ("x6", "y6") if order == 1 else ("x7", "y7")

def _block18_horizon_profile_color(trace_name):
    owner = _block18_benchmark_trace_owner(trace_name)
    if owner is None:
        return block18_asset_profile_color
    owner_index = benchmark_order.index(owner) if owner in benchmark_order else 0
    return block18_benchmark_profile_palette[
        owner_index % len(block18_benchmark_profile_palette)
    ]

def _block18_horizon_profile_hovertemplate(trace_name, profile_kind):
    trace_name = str(trace_name)
    owner = _block18_benchmark_trace_owner(trace_name)
    identity_label = (
        f"Benchmark / index: {owner}"
        if owner is not None else f"Asset (stock / ETF): {ticker_str}"
    )
    metric_label = (
        "Cross-window relative Z-score"
        if trace_name.endswith("Cross-Window Relative Z-Score")
        else "Current Sharpe Z-score"
    )
    return (
        f"<b>{identity_label}</b><br>"
        f"Profile: {profile_kind}<br>"
        "Lookback horizon: %{x} trading days<br>"
        + metric_label + ": %{y:.2f}<extra></extra>"
    )

def _block18_horizon_derivative_hovertemplate(trace_name, order, use_spline):
    owner = _block18_benchmark_trace_owner(trace_name)
    identity_label = (
        f"Benchmark / index: {owner}"
        if owner is not None else f"Asset (stock / ETF): {ticker_str}"
    )
    derivative_label = "Slope" if order == 1 else "Curvature"
    profile_label = "Cubic B-spline" if use_spline else "Raw finite difference"
    units = "Z-score / 20 horizon days" if order == 1 else "Z-score / (20 horizon days)^2"
    return (
        f"<b>{identity_label}</b><br>"
        f"Derivative source: {profile_label}<br>"
        "Lookback horizon: %{x} trading days<br>"
        + derivative_label + ": %{y:.3f} " + units + "<extra></extra>"
    )

def _block18_horizon_extrema_hovertemplate(trace_name, kind, use_spline):
    owner = _block18_benchmark_trace_owner(trace_name)
    identity_label = (
        f"Benchmark / index: {owner}"
        if owner is not None else f"Asset (stock / ETF): {ticker_str}"
    )
    turning_label = "Peak" if kind == "peak" else "Trough"
    curvature_label = "negative" if kind == "peak" else "positive"
    profile_label = "Cubic B-spline" if use_spline else "Raw profile"
    return (
        f"<b>{identity_label}</b><br>"
        f"Turning point: {turning_label}<br>"
        f"Profile: {profile_label}<br>"
        "Lookback horizon: %{x} trading days<br>"
        "Z-score: %{y:.2f}<br>"
        f"Slope crosses zero; curvature is {curvature_label}<extra></extra>"
    )

def _block18_horizon_transition_hovertemplate(trace_name, feature, use_spline):
    owner = _block18_benchmark_trace_owner(trace_name)
    identity_label = (
        f"Benchmark / index: {owner}"
        if owner is not None else f"Asset (stock / ETF): {ticker_str}"
    )
    profile_label = "Cubic B-spline" if use_spline else "Raw finite difference"
    if feature == "movement":
        feature_label = "Fastest curve movement between turning points"
        measure_label = "Slope"
        units = "Z-score / 20 horizon days"
    else:
        feature_label = "Fastest change in rate between turning points"
        measure_label = "Curvature"
        units = "Z-score / (20 horizon days)^2"
    return (
        f"<b>{identity_label}</b><br>"
        f"{feature_label}<br>"
        f"Derivative source: {profile_label}<br>"
        "Lookback horizon: %{x} trading days<br>"
        + measure_label + ": %{y:.3f} " + units + "<extra></extra>"
    )

def _block18_horizon_transition_profile_hovertemplate(trace_name, feature, use_spline):
    owner = _block18_benchmark_trace_owner(trace_name)
    identity_label = (
        f"Benchmark / index: {owner}"
        if owner is not None else f"Asset (stock / ETF): {ticker_str}"
    )
    feature_label = (
        "Fastest curve movement"
        if feature == "movement" else "Fastest change in rate"
    )
    profile_label = "Cubic B-spline" if use_spline else "Raw profile"
    return (
        f"<b>{identity_label}</b><br>"
        f"Profile marker: {feature_label}<br>"
        f"Profile: {profile_label}<br>"
        "Lookback horizon: %{x} trading days<br>"
        "Profile Z-score: %{y:.2f}<extra></extra>"
    )

def _block18_horizon_deceleration_hovertemplate(trace_name, panel, use_spline):
    owner = _block18_benchmark_trace_owner(trace_name)
    identity_label = (
        f"Benchmark / index: {owner}"
        if owner is not None else f"Asset (stock / ETF): {ticker_str}"
    )
    profile_label = "Cubic B-spline" if use_spline else "Raw finite difference"
    value_label = "Profile Z-score" if panel == "profile" else "Slope"
    units = "" if panel == "profile" else " Z-score / 20 horizon days"
    return (
        f"<b>{identity_label}</b><br>"
        "Deceleration leg: |slope| is decreasing<br>"
        f"Derivative source: {profile_label}<br>"
        "Lookback horizon: %{x} trading days<br>"
        + value_label + ": %{y:.3f}" + units + "<extra></extra>"
    )

def _block18_apply_horizon_profile_colors(figure):
    for trace in figure.data:
        trace_name = str(getattr(trace, "name", ""))
        if not _block18_is_current_horizon_profile(trace_name):
            continue
        if getattr(trace, "line", None) is not None:
            trace.line.color = _block18_horizon_profile_color(trace_name)
        trace.hovertemplate = _block18_horizon_profile_hovertemplate(trace_name, "Raw values")
    return figure

def _block18_cubic_bspline_series(series, strength):
    numeric = pd.to_numeric(pd.Series(series), errors="coerce")
    x_values = pd.to_numeric(pd.Series(numeric.index), errors="coerce").to_numpy(dtype=float)
    y_values = numeric.to_numpy(dtype=float)
    finite = np.isfinite(x_values) & np.isfinite(y_values)
    smoothed = np.full(len(numeric), np.nan, dtype=float)
    if finite.sum() < 4:
        smoothed[finite] = y_values[finite]
        return pd.Series(smoothed, index=numeric.index)

    x_fit = x_values[finite]
    y_fit = y_values[finite]
    strength = _block18_normalize_spline_strength(strength) / 100.0
    variance = float(np.nanvar(y_fit))
    smoothing_budget = strength * len(y_fit) * max(variance, 1e-8)
    try:
        spline = UnivariateSpline(x_fit, y_fit, k=3, s=smoothing_budget)
        evaluation_mask = finite & (x_values >= x_fit.min()) & (x_values <= x_fit.max())
        smoothed[evaluation_mask] = spline(x_values[evaluation_mask])
    except Exception:
        smoothed[finite] = y_fit
    return pd.Series(smoothed, index=numeric.index)

def _block18_horizon_derivative_series(series, strength, order, use_spline):
    numeric = pd.to_numeric(pd.Series(series), errors="coerce")
    x_values = pd.to_numeric(pd.Series(numeric.index), errors="coerce").to_numpy(dtype=float)
    y_values = numeric.to_numpy(dtype=float)
    finite = np.isfinite(x_values) & np.isfinite(y_values)
    derivative = np.full(len(numeric), np.nan, dtype=float)
    minimum_points = 4 if use_spline else max(3, order + 1)
    if finite.sum() < minimum_points:
        return pd.Series(derivative, index=numeric.index)

    x_fit = x_values[finite]
    y_fit = y_values[finite]
    try:
        if use_spline:
            normalized_strength = _block18_normalize_spline_strength(strength) / 100.0
            variance = float(np.nanvar(y_fit))
            smoothing_budget = normalized_strength * len(y_fit) * max(variance, 1e-8)
            fitted_spline = UnivariateSpline(x_fit, y_fit, k=3, s=smoothing_budget)
            derivative_values = fitted_spline.derivative(order)(x_fit)
        else:
            derivative_values = y_fit.copy()
            for _ in range(order):
                derivative_values = np.gradient(
                    derivative_values, x_fit, edge_order=2
                )
        derivative[finite] = derivative_values * (block18_horizon_derivative_scale ** order)
    except Exception:
        pass
    return pd.Series(derivative, index=numeric.index)

def _block18_horizon_extrema_series(
    profile_series, first_derivative, second_derivative, kind
):
    profile = pd.to_numeric(pd.Series(profile_series), errors="coerce")
    first = pd.to_numeric(pd.Series(first_derivative), errors="coerce").reindex(profile.index)
    second = pd.to_numeric(pd.Series(second_derivative), errors="coerce").reindex(profile.index)
    markers = pd.Series(np.nan, index=profile.index, dtype=float)
    finite = np.isfinite(profile.to_numpy(dtype=float))
    finite &= np.isfinite(first.to_numpy(dtype=float))
    finite &= np.isfinite(second.to_numpy(dtype=float))
    if finite.sum() < 5:
        return markers

    finite_positions = np.flatnonzero(finite)
    profile_values = profile.iloc[finite_positions].to_numpy(dtype=float)
    first_values = first.iloc[finite_positions].to_numpy(dtype=float)
    second_values = second.iloc[finite_positions].to_numpy(dtype=float)
    robust_range = float(
        np.nanpercentile(profile_values, 95) - np.nanpercentile(profile_values, 5)
    )
    prominence = max(
        block18_extrema_min_prominence,
        block18_extrema_prominence_fraction * max(robust_range, 0.0),
    )
    minimum_separation = max(2, int(round(len(profile_values) * 0.02)))
    direction = 1.0 if kind == "peak" else -1.0
    candidate_positions, _ = find_peaks(
        direction * profile_values,
        prominence=prominence,
        distance=minimum_separation,
    )
    for candidate in candidate_positions:
        if candidate <= 0 or candidate >= len(profile_values) - 1:
            continue
        if kind == "peak":
            derivative_crossing = first_values[candidate - 1] >= 0 >= first_values[candidate + 1]
            curvature_confirms = second_values[candidate] < 0
        else:
            derivative_crossing = first_values[candidate - 1] <= 0 <= first_values[candidate + 1]
            curvature_confirms = second_values[candidate] > 0
        if derivative_crossing and curvature_confirms:
            original_position = finite_positions[candidate]
            markers.iloc[original_position] = profile_values[candidate]
    return markers

def _block18_horizon_transition_marker_series(
    first_derivative, second_derivative, peak_markers, trough_markers, feature
):
    first = pd.to_numeric(pd.Series(first_derivative), errors="coerce")
    second = pd.to_numeric(pd.Series(second_derivative), errors="coerce").reindex(first.index)
    peaks = pd.to_numeric(pd.Series(peak_markers), errors="coerce").reindex(first.index)
    troughs = pd.to_numeric(pd.Series(trough_markers), errors="coerce").reindex(first.index)
    source = first if feature == "movement" else second
    markers = pd.Series(np.nan, index=first.index, dtype=float)
    turning_points = sorted(
        [(int(position), "peak") for position in np.flatnonzero(peaks.notna().to_numpy())]
        + [(int(position), "trough") for position in np.flatnonzero(troughs.notna().to_numpy())]
    )
    source_values = source.to_numpy(dtype=float)
    for (left, left_kind), (right, right_kind) in zip(turning_points, turning_points[1:]):
        if left_kind == right_kind or right - left <= 2:
            continue
        interior = np.arange(left + 1, right, dtype=int)
        finite_interior = interior[np.isfinite(source_values[interior])]
        if finite_interior.size == 0:
            continue
        selected = int(
            finite_interior[np.argmax(np.abs(source_values[finite_interior]))]
        )
        markers.iloc[selected] = source_values[selected]
    return markers

def _block18_project_transition_markers_to_profile(profile_series, transition_markers):
    profile = pd.to_numeric(pd.Series(profile_series), errors="coerce")
    transitions = pd.to_numeric(pd.Series(transition_markers), errors="coerce").reindex(profile.index)
    projected = pd.Series(np.nan, index=profile.index, dtype=float)
    selected = transitions.notna() & profile.notna()
    projected.loc[selected] = profile.loc[selected]
    return projected

def _block18_horizon_deceleration_leg_series(
    source_series, peak_markers, trough_markers, movement_markers
):
    source = pd.to_numeric(pd.Series(source_series), errors="coerce")
    peaks = pd.to_numeric(pd.Series(peak_markers), errors="coerce").reindex(source.index)
    troughs = pd.to_numeric(pd.Series(trough_markers), errors="coerce").reindex(source.index)
    movement = pd.to_numeric(pd.Series(movement_markers), errors="coerce").reindex(source.index)
    deceleration = pd.Series(np.nan, index=source.index, dtype=float)
    turning_points = sorted(
        [(int(position), "peak") for position in np.flatnonzero(peaks.notna().to_numpy())]
        + [(int(position), "trough") for position in np.flatnonzero(troughs.notna().to_numpy())]
    )
    movement_positions = np.flatnonzero(movement.notna().to_numpy())
    for (left, left_kind), (right, right_kind) in zip(turning_points, turning_points[1:]):
        if left_kind == right_kind:
            continue
        interval_movements = movement_positions[
            (movement_positions > left) & (movement_positions < right)
        ]
        if interval_movements.size == 0:
            continue
        maximum_speed_position = int(interval_movements[0])
        deceleration.iloc[maximum_speed_position:right] = (
            source.iloc[maximum_speed_position:right]
        )
    return deceleration

def _block18_add_spline_updates(trace_updates, spline_toggle, spline_strength):
    if not _block18_spline_is_enabled(spline_toggle):
        return trace_updates
    for trace_name, values in list(trace_updates.items()):
        if _block18_is_current_horizon_profile(trace_name):
            trace_updates[_block18_spline_trace_name(trace_name)] = _block18_cubic_bspline_series(
                values, spline_strength
            )
    return trace_updates

def _block18_add_horizon_derivative_updates(
    trace_updates, spline_toggle, spline_strength
):
    use_spline = _block18_spline_is_enabled(spline_toggle)
    for trace_name, values in list(trace_updates.items()):
        if not _block18_is_current_horizon_profile(trace_name):
            continue
        for order in (1, 2):
            derivative_name = _block18_horizon_derivative_trace_name(
                trace_name, order, use_spline
            )
            trace_updates[derivative_name] = _block18_horizon_derivative_series(
                values, spline_strength, order, use_spline
            )
    return trace_updates

def _block18_add_horizon_extrema_updates(
    trace_updates, spline_toggle
):
    use_spline = _block18_spline_is_enabled(spline_toggle)
    raw_updates = [
        (trace_name, values) for trace_name, values in list(trace_updates.items())
        if _block18_is_current_horizon_profile(trace_name)
    ]
    for trace_name, raw_values in raw_updates:
        profile_name = _block18_spline_trace_name(trace_name) if use_spline else trace_name
        profile_values = trace_updates.get(profile_name, raw_values)
        first_derivative = trace_updates.get(
            _block18_horizon_derivative_trace_name(trace_name, 1, use_spline)
        )
        second_derivative = trace_updates.get(
            _block18_horizon_derivative_trace_name(trace_name, 2, use_spline)
        )
        if first_derivative is None or second_derivative is None:
            continue
        for kind in ("peak", "trough"):
            trace_updates[_block18_horizon_extrema_trace_name(
                trace_name, kind, use_spline
            )] = _block18_horizon_extrema_series(
                profile_values, first_derivative, second_derivative, kind
            )
    return trace_updates

def _block18_add_horizon_transition_updates(trace_updates, spline_toggle):
    use_spline = _block18_spline_is_enabled(spline_toggle)
    raw_trace_names = [
        trace_name for trace_name in list(trace_updates)
        if _block18_is_current_horizon_profile(trace_name)
    ]
    for trace_name in raw_trace_names:
        profile_name = _block18_spline_trace_name(trace_name) if use_spline else trace_name
        profile_values = trace_updates.get(profile_name, trace_updates[trace_name])
        first_derivative = trace_updates.get(
            _block18_horizon_derivative_trace_name(trace_name, 1, use_spline)
        )
        second_derivative = trace_updates.get(
            _block18_horizon_derivative_trace_name(trace_name, 2, use_spline)
        )
        peak_markers = trace_updates.get(
            _block18_horizon_extrema_trace_name(trace_name, "peak", use_spline)
        )
        trough_markers = trace_updates.get(
            _block18_horizon_extrema_trace_name(trace_name, "trough", use_spline)
        )
        if any(value is None for value in (
            first_derivative, second_derivative, peak_markers, trough_markers
        )):
            continue
        for feature in ("movement",):
            transition_markers = _block18_horizon_transition_marker_series(
                first_derivative, second_derivative,
                peak_markers, trough_markers, feature,
            )
            trace_updates[_block18_horizon_transition_trace_name(
                trace_name, feature, use_spline
            )] = transition_markers
            trace_updates[_block18_horizon_transition_profile_trace_name(
                trace_name, feature, use_spline
            )] = _block18_project_transition_markers_to_profile(
                profile_values, transition_markers
            )
        movement_markers = trace_updates.get(
            _block18_horizon_transition_trace_name(
                trace_name, "movement", use_spline
            )
        )
        if movement_markers is not None:
            for panel, source_values in (
                ("profile", profile_values), ("slope", first_derivative)
            ):
                trace_updates[_block18_horizon_deceleration_trace_name(
                    trace_name, panel, use_spline
                )] = _block18_horizon_deceleration_leg_series(
                    source_values, peak_markers, trough_markers, movement_markers
                )
    return trace_updates

def _block18_configure_spline_traces(
    figure, spline_toggle, spline_strength, raw_overlay_toggle=None, *, compute_values=True
):
    figure = _block18_apply_horizon_profile_colors(figure)
    figure.data = tuple(
        trace for trace in figure.data
        if not _block18_is_horizon_derivative_trace(getattr(trace, "name", ""))
        and not _block18_is_horizon_extrema_trace(getattr(trace, "name", ""))
        and not _block18_is_horizon_transition_trace(getattr(trace, "name", ""))
        and not _block18_is_horizon_deceleration_trace(getattr(trace, "name", ""))
    )
    use_spline = _block18_spline_is_enabled(spline_toggle)
    show_raw_overlay = _block18_raw_overlay_is_enabled(raw_overlay_toggle)
    raw_profile_traces = [
        trace for trace in figure.data
        if _block18_is_current_horizon_profile(getattr(trace, "name", ""))
    ]
    for raw_trace in raw_profile_traces:
        raw_series = pd.Series(
            list(raw_trace.y), index=pd.to_numeric(pd.Series(list(raw_trace.x)), errors="coerce")
        )
        spline_line = raw_trace.line.to_plotly_json() if getattr(raw_trace, "line", None) is not None else {}
        profile_color = spline_line.get("color") or _block18_horizon_profile_color(raw_trace.name)
        active_profile = raw_series
        if use_spline:
            smoothed = (
                _block18_cubic_bspline_series(raw_series, spline_strength)
                if compute_values else pd.Series(np.nan, index=raw_series.index)
            )
            active_profile = smoothed
            spline_line.update({
                "color": profile_color,
                "width": max(float(spline_line.get("width") or 2.0) + 1.5, 3.5),
                "dash": "solid",
            })
            if show_raw_overlay:
                raw_trace.line.color = profile_color
                raw_trace.line.width = 1.25
                raw_trace.line.dash = "solid"
                raw_trace.opacity = 0.40
                raw_trace.showlegend = False
            figure.add_trace(go.Scatter(
                x=list(raw_trace.x),
                y=_block18_json_values(smoothed),
                mode="lines",
                name=_block18_spline_trace_name(raw_trace.name),
                line=spline_line,
                opacity=1.0,
                showlegend=True,
                xaxis=raw_trace.xaxis,
                yaxis=raw_trace.yaxis,
                hovertemplate=_block18_horizon_profile_hovertemplate(
                    raw_trace.name, "Cubic B-spline"
                ),
            ))
        derivatives = {}
        for order in (1, 2):
            derivative = (
                _block18_horizon_derivative_series(
                    raw_series, spline_strength, order, use_spline
                )
                if compute_values else pd.Series(np.nan, index=raw_series.index)
            )
            derivatives[order] = derivative
            derivative_xaxis, derivative_yaxis = _block18_horizon_derivative_axes(
                raw_trace.name, order
            )
            figure.add_trace(go.Scatter(
                x=list(raw_trace.x),
                y=_block18_json_values(derivative),
                mode="lines",
                name=_block18_horizon_derivative_trace_name(
                    raw_trace.name, order, use_spline
                ),
                line={"color": profile_color, "width": 2.4, "dash": "solid"},
                opacity=1.0,
                showlegend=False,
                xaxis=derivative_xaxis,
                yaxis=derivative_yaxis,
                hovertemplate=_block18_horizon_derivative_hovertemplate(
                    raw_trace.name, order, use_spline
                ),
            ))
        extrema_by_kind = {}
        for kind in ("peak", "trough"):
            extrema = _block18_horizon_extrema_series(
                active_profile, derivatives[1], derivatives[2], kind
            )
            extrema_by_kind[kind] = extrema
            marker_symbol = "circle"
            marker_color = "#22c55e" if kind == "peak" else "#ef4444"
            figure.add_trace(go.Scatter(
                x=list(raw_trace.x),
                y=_block18_json_values(extrema),
                mode="markers",
                name=_block18_horizon_extrema_trace_name(
                    raw_trace.name, kind, use_spline
                ),
                marker={
                    "color": marker_color,
                    "size": 18,
                    "symbol": marker_symbol,
                    "line": {"color": "#0b0f14", "width": 3.5},
                },
                opacity=1.0,
                showlegend=False,
                cliponaxis=False,
                xaxis=raw_trace.xaxis,
                yaxis=raw_trace.yaxis,
                hovertemplate=_block18_horizon_extrema_hovertemplate(
                    raw_trace.name, kind, use_spline
                ),
            ))
        movement_markers = _block18_horizon_transition_marker_series(
            derivatives[1], derivatives[2],
            extrema_by_kind["peak"], extrema_by_kind["trough"], "movement",
        )
        first_derivative_xaxis, first_derivative_yaxis = (
            _block18_horizon_derivative_axes(raw_trace.name, 1)
        )
        for panel, source_values, panel_xaxis, panel_yaxis in (
            ("profile", active_profile, raw_trace.xaxis, raw_trace.yaxis),
            ("slope", derivatives[1], first_derivative_xaxis, first_derivative_yaxis),
        ):
            deceleration_leg = _block18_horizon_deceleration_leg_series(
                source_values, extrema_by_kind["peak"],
                extrema_by_kind["trough"], movement_markers,
            )
            figure.add_trace(go.Scatter(
                x=list(raw_trace.x),
                y=_block18_json_values(deceleration_leg),
                mode="lines",
                name=_block18_horizon_deceleration_trace_name(
                    raw_trace.name, panel, use_spline
                ),
                line={"color": profile_color, "width": 6.0, "dash": "dot"},
                opacity=0.90,
                showlegend=False,
                connectgaps=False,
                xaxis=panel_xaxis,
                yaxis=panel_yaxis,
                hovertemplate=_block18_horizon_deceleration_hovertemplate(
                    raw_trace.name, panel, use_spline
                ),
            ))
        transition_specs = {
            "movement": {"order": 1, "symbol": "diamond", "size": 16},
        }
        for feature, marker_spec in transition_specs.items():
            transition_markers = movement_markers
            derivative_xaxis, derivative_yaxis = _block18_horizon_derivative_axes(
                raw_trace.name, marker_spec["order"]
            )
            figure.add_trace(go.Scatter(
                x=list(raw_trace.x),
                y=_block18_json_values(transition_markers),
                mode="markers",
                name=_block18_horizon_transition_trace_name(
                    raw_trace.name, feature, use_spline
                ),
                marker={
                    "color": profile_color,
                    "size": marker_spec["size"],
                    "symbol": marker_spec["symbol"],
                    "line": {"color": "#0b0f14", "width": 3.5},
                },
                opacity=1.0,
                showlegend=False,
                cliponaxis=False,
                xaxis=derivative_xaxis,
                yaxis=derivative_yaxis,
                hovertemplate=_block18_horizon_transition_hovertemplate(
                    raw_trace.name, feature, use_spline
                ),
            ))
            profile_transition_markers = _block18_project_transition_markers_to_profile(
                active_profile, transition_markers
            )
            figure.add_trace(go.Scatter(
                x=list(raw_trace.x),
                y=_block18_json_values(profile_transition_markers),
                mode="markers",
                name=_block18_horizon_transition_profile_trace_name(
                    raw_trace.name, feature, use_spline
                ),
                marker={
                    "color": profile_color,
                    "size": marker_spec["size"],
                    "symbol": marker_spec["symbol"],
                    "line": {"color": "#0b0f14", "width": 3.5},
                },
                opacity=1.0,
                showlegend=False,
                cliponaxis=False,
                xaxis=raw_trace.xaxis,
                yaxis=raw_trace.yaxis,
                hovertemplate=_block18_horizon_transition_profile_hovertemplate(
                    raw_trace.name, feature, use_spline
                ),
            ))
    if use_spline and not show_raw_overlay:
        figure.data = tuple(
            trace for trace in figure.data
            if not _block18_is_current_horizon_profile(getattr(trace, "name", ""))
        )
    return figure

def _block18_prepare_window_diagnostics_playback():
    if block18_window_diagnostics_playback_cache:
        return block18_window_diagnostics_playback_cache

    for symbol, diagnostics_context in momentum_diagnostics_contexts.items():
        sharpe_frame = pd.DataFrame(diagnostics_context["sharpe_table"]).copy()
        sharpe_frame = sharpe_frame.apply(pd.to_numeric, errors="coerce").sort_index()
        sharpe_frame.index = pd.to_datetime(
            sharpe_frame.index, errors="coerce", utc=True
        ).tz_convert(None).normalize()
        sharpe_frame = sharpe_frame.loc[~sharpe_frame.index.isna()]
        sharpe_frame = sharpe_frame.loc[~sharpe_frame.index.duplicated(keep="last")]

        expanding_mean = sharpe_frame.expanding(min_periods=2).mean()
        expanding_std = sharpe_frame.expanding(min_periods=2).std().replace(0.0, np.nan)
        sharpe_zscore = sharpe_frame.sub(expanding_mean).div(expanding_std).astype("float32")
        cross_window_mean = sharpe_zscore.mean(axis=1)
        cross_window_std = sharpe_zscore.std(axis=1).replace(0.0, np.nan)
        cross_window_zscore = sharpe_zscore.sub(cross_window_mean, axis=0).div(
            cross_window_std, axis=0
        ).astype("float32")

        playback_context = {
            "sharpe_zscore": sharpe_zscore,
            "cross_window_zscore": cross_window_zscore,
        }
        if symbol == ticker_str:
            playback_context.update({
                "sharpe_reference_mean": sharpe_zscore.expanding(min_periods=2).mean().astype("float32"),
                "sharpe_reference_std": sharpe_zscore.expanding(min_periods=2).std().astype("float32"),
                "cross_reference_mean": cross_window_zscore.expanding(min_periods=2).mean().astype("float32"),
                "cross_reference_std": cross_window_zscore.expanding(min_periods=2).std().astype("float32"),
            })
        block18_window_diagnostics_playback_cache[symbol] = playback_context

    return block18_window_diagnostics_playback_cache

def _block18_diagnostics_asof_row(frame, as_of_date):
    if frame.empty:
        return pd.Series(dtype=float)
    row_position = frame.index.searchsorted(pd.Timestamp(as_of_date), side="right") - 1
    if row_position < 0:
        return pd.Series(index=frame.columns, dtype=float)
    return pd.to_numeric(frame.iloc[row_position], errors="coerce")

def _block18_window_diagnostics_trace_updates(
    playback_position, selected_benchmarks, spline_toggle=None, spline_strength=None
):
    playback_contexts = _block18_prepare_window_diagnostics_playback()
    as_of_date, playback_position = _block18_playback_date(playback_position)
    asset_context = playback_contexts[ticker_str]

    current_sharpe = _block18_diagnostics_asof_row(
        asset_context["sharpe_zscore"], as_of_date
    )
    current_cross = _block18_diagnostics_asof_row(
        asset_context["cross_window_zscore"], as_of_date
    )
    sharpe_mean = _block18_diagnostics_asof_row(
        asset_context["sharpe_reference_mean"], as_of_date
    )
    sharpe_std = _block18_diagnostics_asof_row(
        asset_context["sharpe_reference_std"], as_of_date
    )
    cross_mean = _block18_diagnostics_asof_row(
        asset_context["cross_reference_mean"], as_of_date
    )
    cross_std = _block18_diagnostics_asof_row(
        asset_context["cross_reference_std"], as_of_date
    )

    trace_updates = {
        "Current Sharpe Z-Score": current_sharpe,
        "Cross-Window Relative Z-Score": current_cross,
        "Historical Mean Sharpe Z-Score": sharpe_mean,
        "Historical Mean Cross-Window Relative Z-Score": cross_mean,
    }
    for level in (1, 2):
        trace_updates[f"Historical Sharpe Z-Score +{level} Std Dev"] = sharpe_mean + level * sharpe_std
        trace_updates[f"Historical Sharpe Z-Score -{level} Std Dev"] = sharpe_mean - level * sharpe_std
        trace_updates[f"Historical Cross-Window Relative Z-Score +{level} Std Dev"] = cross_mean + level * cross_std
        trace_updates[f"Historical Cross-Window Relative Z-Score -{level} Std Dev"] = cross_mean - level * cross_std

    for symbol in selected_benchmarks:
        benchmark_context = playback_contexts.get(symbol)
        if benchmark_context is None:
            continue
        trace_updates[f"{symbol} Current Sharpe Z-Score"] = _block18_diagnostics_asof_row(
            benchmark_context["sharpe_zscore"], as_of_date
        )
        trace_updates[f"{symbol} Cross-Window Relative Z-Score"] = _block18_diagnostics_asof_row(
            benchmark_context["cross_window_zscore"], as_of_date
        )
    trace_updates = _block18_add_spline_updates(
        trace_updates, spline_toggle, spline_strength
    )
    trace_updates = _block18_add_horizon_derivative_updates(
        trace_updates, spline_toggle, spline_strength
    )
    trace_updates = _block18_add_horizon_extrema_updates(
        trace_updates, spline_toggle
    )
    trace_updates = _block18_add_horizon_transition_updates(
        trace_updates, spline_toggle
    )
    return trace_updates, as_of_date, playback_position

def _block18_json_values(series):
    return [None if pd.isna(value) else float(value) for value in pd.Series(series)]

block18_diagnostics_playback_titles = (
    "Current Sharpe Z-Score by Window",
    "First Horizon Derivative of Current Sharpe Z-Score",
    "Second Horizon Derivative of Current Sharpe Z-Score",
    "Cross-Window Relative Z-Score by Window",
    "First Horizon Derivative of Cross-Window Relative Z-Score",
    "Second Horizon Derivative of Cross-Window Relative Z-Score",
)

def _block18_diagnostics_annotation_for_date(annotation_text, as_of_date):
    annotation_text = str(annotation_text)
    for title in block18_diagnostics_playback_titles:
        if annotation_text.startswith(title):
            return f"{title} ({as_of_date:%Y-%m-%d})"
    return None

def _block18_apply_window_diagnostics_asof(
    figure, playback_position, selected_benchmarks, spline_toggle=None, spline_strength=None,
    raw_overlay_toggle=None,
):
    trace_updates, as_of_date, playback_position = _block18_window_diagnostics_trace_updates(
        playback_position, selected_benchmarks, spline_toggle, spline_strength
    )
    for trace in figure.data:
        trace_name = str(getattr(trace, "name", ""))
        if trace_name in trace_updates:
            trace.y = _block18_json_values(trace_updates[trace_name])
    for annotation in figure.layout.annotations or []:
        annotation_text = str(getattr(annotation, "text", ""))
        updated_text = _block18_diagnostics_annotation_for_date(
            annotation_text, as_of_date
        )
        if updated_text is not None:
            annotation.text = updated_text
    figure = _block18_configure_spline_traces(
        figure, spline_toggle, spline_strength, raw_overlay_toggle
    )
    return figure, as_of_date

def _block18_patch_window_diagnostics(
    playback_position, selected_benchmarks, spline_toggle=None, spline_strength=None,
    raw_overlay_toggle=None,
):
    trace_updates, as_of_date, playback_position = _block18_window_diagnostics_trace_updates(
        playback_position, selected_benchmarks, spline_toggle, spline_strength
    )
    source_figure = _block18_filter_benchmark_traces(
        _block18_named_figure(
            "fig_momentum_window_diagnostics_grid",
            "Functional Horizon Profile",
            default_height=2750,
        ),
        selected_benchmarks,
    )
    source_figure = _block18_configure_spline_traces(
        source_figure, spline_toggle, spline_strength, raw_overlay_toggle,
        compute_values=False,
    )
    patched_figure = Patch()
    for trace_index, trace in enumerate(source_figure.data):
        trace_name = str(getattr(trace, "name", ""))
        if trace_name in trace_updates:
            patched_figure["data"][trace_index]["y"] = _block18_json_values(
                trace_updates[trace_name]
            )
    for annotation_index, annotation in enumerate(source_figure.layout.annotations or []):
        annotation_text = str(getattr(annotation, "text", ""))
        updated_text = _block18_diagnostics_annotation_for_date(
            annotation_text, as_of_date
        )
        if updated_text is not None:
            patched_figure["layout"]["annotations"][annotation_index]["text"] = updated_text
    return patched_figure, as_of_date

def _block18_playback_date(position):
    try:
        position = int(position)
    except (TypeError, ValueError):
        position = block18_default_playback_position
    position = max(0, min(position, len(block18_playback_dates) - 1))
    return pd.Timestamp(block18_playback_dates[position]), position

def _block18_playback_position_for_date(selected_date):
    try:
        selected_date = pd.Timestamp(selected_date)
        if pd.isna(selected_date):
            return block18_default_playback_position
        if selected_date.tzinfo is not None:
            selected_date = selected_date.tz_convert(None)
        selected_date = selected_date.normalize()
    except (TypeError, ValueError):
        return block18_default_playback_position
    position = int(block18_playback_dates.searchsorted(selected_date, side="right") - 1)
    return max(0, min(position, len(block18_playback_dates) - 1))

def _block18_cached_tab_figure(
    active_tab, window, display_range_value, selected_benchmarks, cache_version=0
):
    cache_key = (
        active_tab,
        int(window),
        display_range_value,
        tuple(selected_benchmarks),
        int(cache_version or 0),
    )
    if cache_key not in block18_base_figure_cache:
        if len(block18_base_figure_cache) >= 24:
            block18_base_figure_cache.clear()
        block18_base_figure_cache[cache_key] = _block18_build_tab_figure(
            active_tab, window, display_range_value, selected_benchmarks
        )
    return go.Figure(block18_base_figure_cache[cache_key])

def _block18_next_playback_state(trigger_id, playing, position, active_tab):
    maximum = block18_default_playback_position
    _, position = _block18_playback_date(position)
    playing = bool(playing)

    if active_tab != "window_diagnostics":
        return False, True, "Play", position
    if trigger_id == "diagnostics-step-back-button":
        return False, True, "Play", max(0, position - 1)
    if trigger_id == "diagnostics-step-forward-button":
        next_position = min(position + 1, maximum)
        return False, True, "Replay" if next_position >= maximum else "Play", next_position
    if trigger_id == "diagnostics-playback-button":
        playing = not playing
        if playing and position >= maximum:
            position = 0
    elif trigger_id == "diagnostics-playback-interval" and playing:
        position = min(position + block18_playback_step, maximum)
        if position >= maximum:
            return False, True, "Replay", position

    return playing, not playing, "Pause" if playing else "Play", position

def _block18_render_dashboard_figure(
    active_tab,
    window_value,
    display_range_value,
    selected_benchmarks=None,
    playback_position=None,
    diagnostics_window_range=None,
    spline_toggle=None,
    spline_strength=None,
    raw_overlay_toggle=None,
    cache_version=0,
):
    active_tab = active_tab if active_tab in block18_tab_config else "risk"
    display_range_value = display_range_value if display_range_value in block18_display_range_offsets else block18_default_display_range
    selected_benchmarks = _block18_normalize_benchmark_selection(selected_benchmarks)
    label = block18_tab_config[active_tab]["label"]
    display_label = _block18_display_range_label(display_range_value)

    try:
        _block18_validate_peer_state(active_tab, required_benchmarks=selected_benchmarks)
        if active_tab in {"risk", "upside", "drawdown", "correlation"}:
            window = _block18_validate_window(window_value)
            figure = _block18_cached_tab_figure(
                active_tab, window, display_range_value, selected_benchmarks, cache_version
            )
            status = f"{label} | Window: {window:,} trading days | Display: {display_label}"
        else:
            figure = _block18_cached_tab_figure(
                active_tab, block18_default_window, display_range_value, selected_benchmarks, cache_version
            )
            status = f"{label} | Display: {display_label}" if active_tab in block18_date_range_tabs else f"{label} | {block18_native_axis_status.get(active_tab, 'Native axes')}"
        _block18_validate_peer_state(active_tab, figure, selected_benchmarks)
        playback_date, playback_position = _block18_playback_date(playback_position)
        if active_tab == "window_diagnostics":
            figure, playback_date = _block18_apply_window_diagnostics_asof(
                figure, playback_position, selected_benchmarks, spline_toggle, spline_strength,
                raw_overlay_toggle,
            )
            figure, diagnostics_window_range = _block18_apply_diagnostics_window_range(
                figure, diagnostics_window_range
            )
        if active_tab in block18_date_range_tabs:
            figure = _block18_apply_display_range(
                figure,
                display_range_value,
                preserve_benchmark_selector=active_tab == "heatmap",
            )
        else:
            figure = _block18_strip_figure_controls(figure)
        if active_tab in {"heatmap", "window_diagnostics", "risk", "correlation"}:
            status += " | Benchmarks: " + _block18_benchmark_selection_label(selected_benchmarks)
        if active_tab == "window_diagnostics":
            status += (
                f" | Windows: {diagnostics_window_range[0]}-{diagnostics_window_range[1]} days"
                + (
                    f" | Cubic B-spline: {_block18_normalize_spline_strength(spline_strength):.0f}%"
                    if _block18_spline_is_enabled(spline_toggle) else " | Cubic B-spline: Off"
                )
                + (
                    " | Raw overlay: On"
                    if _block18_spline_is_enabled(spline_toggle)
                    and _block18_raw_overlay_is_enabled(raw_overlay_toggle) else ""
                )
                + f" | As of: {playback_date:%Y-%m-%d}"
            )
        return figure, status
    except Exception as error:
        message = f"{label} could not render: {error}"
        return _block18_error_figure(message, height=block18_tab_config[active_tab]["height"]), message

block18_initial_figure, block18_initial_status = _block18_render_dashboard_figure(
    "heatmap",
    block18_default_window,
    block18_default_display_range,
    block18_default_benchmark_selection,
)

block18_controls = html.Div(
    [
        html.Div(
            [
                html.Div("Window", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                dcc.Input(
                    id="shared-window-input",
                    type="number",
                    min=2,
                    max=10000,
                    step=1,
                    value=block18_default_window,
                    debounce=True,
                    style={
                        "width": "130px",
                        "height": "36px",
                        "backgroundColor": "#111827",
                        "border": "1px solid #334155",
                        "color": "#f8fafc",
                        "padding": "0 10px",
                    },
                ),
            ]
        ),
        html.Div(
            [
                html.Div("Lookback", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                dcc.Dropdown(
                    id="shared-display-range-dropdown",
                    options=block18_display_range_options,
                    value=block18_default_display_range,
                    clearable=False,
                    style={"width": "150px", "color": "#0f172a"},
                ),
            ]
        ),
        html.Div(
            [
                html.Div("Benchmark indices", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                dcc.Dropdown(
                    id="benchmark-index-dropdown",
                    options=[{"label": symbol, "value": symbol} for symbol in benchmark_order],
                    value=block18_default_benchmark_selection,
                    multi=True,
                    clearable=False,
                    placeholder="Select benchmark indices",
                    style={"width": "520px", "color": "#0f172a"},
                ),
            ]
        ),
        html.Button(
            "Update",
            id="shared-update-button",
            n_clicks=0,
            style={
                "height": "38px",
                "alignSelf": "end",
                "backgroundColor": "#2563eb",
                "border": "1px solid #1d4ed8",
                "color": "#ffffff",
                "padding": "0 16px",
            },
        ),
        html.Div(
            id="shared-view-status",
            children=block18_initial_status,
            style={"color": "#cbd5e1", "alignSelf": "end", "paddingBottom": "9px"},
        ),
    ],
    style={
        "display": "flex",
        "gap": "12px",
        "alignItems": "stretch",
        "flexWrap": "wrap",
        "marginBottom": "12px",
    },
)

block18_playback_controls_style = {
    "display": "none",
    "gap": "14px",
    "alignItems": "stretch",
    "flexWrap": "wrap",
    "marginTop": "16px",
    "padding": "10px 12px",
    "border": "1px solid #1e293b",
    "borderRadius": "6px",
}

block18_playback_controls = html.Div(
    [
        html.Div(
            [
                html.Div("Functional horizon playback", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                html.Button(
                    "Play",
                    id="diagnostics-playback-button",
                    n_clicks=0,
                    style={
                        "height": "36px",
                        "width": "78px",
                        "backgroundColor": "#0f766e",
                        "border": "1px solid #14b8a6",
                        "color": "#ffffff",
                    },
                ),
            ]
        ),
        html.Div(
            [
                html.Div("Speed", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                dcc.Dropdown(
                    id="diagnostics-playback-speed-dropdown",
                    options=block18_playback_speed_options,
                    value=block18_default_playback_speed,
                    clearable=False,
                    style={"width": "120px", "color": "#0f172a"},
                ),
            ]
        ),
        html.Div(
            [
                html.Div("Jump to date", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                dcc.DatePickerSingle(
                    id="diagnostics-date-picker",
                    min_date_allowed=block18_playback_dates[0].date(),
                    max_date_allowed=block18_playback_dates[-1].date(),
                    date=block18_playback_dates[block18_default_playback_position].date(),
                    display_format="YYYY-MM-DD",
                    clearable=False,
                    first_day_of_week=0,
                    style={"fontSize": "13px"},
                ),
            ]
        ),
        html.Div(
            [
                html.Div("As-of date (profile and derivative rows)", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                html.Div(
                    [
                        html.Button(
                            "←",
                            id="diagnostics-step-back-button",
                            n_clicks=0,
                            title="Previous trading date",
                            style={"width": "38px", "height": "32px", "fontSize": "20px", "backgroundColor": "#1e293b", "border": "1px solid #475569", "color": "#f8fafc"},
                        ),
                        html.Div(
                            dcc.Slider(
                                id="diagnostics-asof-slider",
                                min=0,
                                max=len(block18_playback_dates) - 1,
                                step=1,
                                value=block18_default_playback_position,
                                marks=block18_playback_marks,
                                included=False,
                                updatemode="drag",
                            ),
                            style={"flex": "1 1 auto", "minWidth": "300px", "padding": "0 8px"},
                        ),
                        html.Button(
                            "→",
                            id="diagnostics-step-forward-button",
                            n_clicks=0,
                            title="Next trading date",
                            style={"width": "38px", "height": "32px", "fontSize": "20px", "backgroundColor": "#1e293b", "border": "1px solid #475569", "color": "#f8fafc"},
                        ),
                    ],
                    style={"display": "flex", "alignItems": "center", "width": "100%"},
                ),
            ],
            style={"flex": "1 1 680px", "minWidth": "420px"},
        ),
        html.Div(
            [
                html.Div("Lookback horizon range shown in profile and derivative rows", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                dcc.RangeSlider(
                    id="diagnostics-window-range-slider",
                    min=block18_diagnostics_window_min,
                    max=block18_diagnostics_window_max,
                    step=1,
                    value=block18_default_diagnostics_window_range,
                    marks=block18_diagnostics_window_marks,
                    allowCross=False,
                    pushable=1,
                    updatemode="drag",
                    tooltip={"placement": "bottom", "always_visible": False},
                ),
            ],
            style={"flex": "1 1 100%", "minWidth": "420px", "paddingTop": "4px"},
        ),
        html.Div(
            [
                html.Div(
                    [
                        html.Div("Profile display", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                        dcc.Checklist(
                            id="diagnostics-spline-toggle",
                            options=[{"label": " Use cubic B-spline profiles", "value": "cubic_bspline"}],
                            value=block18_default_spline_toggle,
                            inline=True,
                            style={"color": "#e2e8f0", "whiteSpace": "nowrap"},
                        ),
                        dcc.Checklist(
                            id="diagnostics-raw-overlay-toggle",
                            options=[{"label": " Overlay raw values", "value": "raw_overlay"}],
                            value=block18_default_raw_overlay_toggle,
                            inline=True,
                            style={"color": "#e2e8f0", "whiteSpace": "nowrap", "marginTop": "4px"},
                        ),
                    ],
                    style={"flex": "0 0 215px"},
                ),
                html.Div(
                    [
                        html.Div("B-spline smoothing strength", style={"fontSize": "11px", "color": "#94a3b8", "marginBottom": "4px"}),
                        dcc.Slider(
                            id="diagnostics-spline-strength-slider",
                            min=0,
                            max=100,
                            step=1,
                            value=block18_default_spline_strength,
                            marks={0: "Exact", 25: "Light", 50: "Medium", 75: "Strong", 100: "Max"},
                            updatemode="mouseup",
                            tooltip={"placement": "bottom", "always_visible": False},
                        ),
                    ],
                    style={"flex": "1 1 520px", "minWidth": "360px"},
                ),
            ],
            style={"display": "flex", "gap": "18px", "alignItems": "center", "flex": "1 1 100%", "paddingTop": "8px"},
        ),
    ],
    id="diagnostics-playback-controls",
    style=block18_playback_controls_style,
)

block18_dash_app = Dash(__name__)
block18_dash_app.layout = html.Div(
    [
        block18_controls,
        dcc.Interval(
            id="diagnostics-playback-interval",
            interval=block18_default_playback_interval,
            n_intervals=0,
            disabled=True,
        ),
        dcc.Store(id="diagnostics-playback-playing", data=False),
        dcc.Tabs(
            id="momentum-efficiency-view-tabs",
            value="heatmap",
            children=[_block18_tab(tab_value) for tab_value in block18_tab_order],
            colors={"border": "#334155", "primary": "#60a5fa", "background": "#111827"},
        ),
        dcc.Graph(
            id="momentum-efficiency-tab-graph",
            figure=block18_initial_figure,
            animate=False,
            animation_options=_block18_animation_options(block18_default_playback_speed),
            config={"responsive": True, "displaylogo": False},
            style={"height": _block18_graph_height(block18_initial_figure, 2150)},
        ),
        block18_playback_controls,
    ],
    style={"backgroundColor": "#0b0f14", "padding": "12px", "color": "#f8fafc"},
)

@block18_dash_app.callback(
    Output("diagnostics-playback-interval", "interval"),
    Output("momentum-efficiency-tab-graph", "animation_options"),
    Input("diagnostics-playback-speed-dropdown", "value"),
)
def _update_diagnostics_playback_speed(speed_milliseconds):
    selected_speed = speed_milliseconds
    if selected_speed == "continuous":
        interval = block18_continuous_playback_interval
    else:
        try:
            interval = max(100, int(selected_speed))
        except (TypeError, ValueError):
            selected_speed = block18_default_playback_speed
            interval = block18_default_playback_interval
    return interval, _block18_animation_options(selected_speed)

@block18_dash_app.callback(
    Output("diagnostics-playback-playing", "data"),
    Output("diagnostics-playback-interval", "disabled"),
    Output("diagnostics-playback-button", "children"),
    Output("diagnostics-asof-slider", "value"),
    Output("diagnostics-playback-controls", "style"),
    Input("diagnostics-playback-button", "n_clicks"),
    Input("diagnostics-playback-interval", "n_intervals"),
    Input("diagnostics-step-back-button", "n_clicks"),
    Input("diagnostics-step-forward-button", "n_clicks"),
    Input("diagnostics-date-picker", "date"),
    Input("momentum-efficiency-view-tabs", "value"),
    Input("benchmark-index-dropdown", "value"),
    Input("diagnostics-window-range-slider", "value"),
    Input("diagnostics-spline-toggle", "value"),
    Input("diagnostics-raw-overlay-toggle", "value"),
    Input("diagnostics-spline-strength-slider", "value"),
    State("diagnostics-playback-playing", "data"),
    State("diagnostics-asof-slider", "value"),
)
def _update_diagnostics_playback(
    _n_clicks, _n_intervals, _step_back_clicks, _step_forward_clicks, selected_date, active_tab, _selected_benchmarks, _window_range, _spline_toggle, _raw_overlay_toggle, _spline_strength, playing, position
):
    if ctx.triggered_id in {
        "benchmark-index-dropdown",
        "diagnostics-window-range-slider",
        "diagnostics-spline-toggle",
        "diagnostics-raw-overlay-toggle",
        "diagnostics-spline-strength-slider",
        "diagnostics-date-picker",
    }:
        playing = False
    if ctx.triggered_id == "diagnostics-date-picker":
        position = _block18_playback_position_for_date(selected_date)
    next_playing, interval_disabled, button_label, next_position = _block18_next_playback_state(
        ctx.triggered_id, playing, position, active_tab
    )
    slider_value = (
        next_position
        if ctx.triggered_id in {
            "diagnostics-playback-button",
            "diagnostics-playback-interval",
            "diagnostics-step-back-button",
            "diagnostics-step-forward-button",
            "diagnostics-date-picker",
        }
        else no_update
    )
    controls_style = {
        **block18_playback_controls_style,
        "display": "flex" if active_tab == "window_diagnostics" else "none",
    }
    return next_playing, interval_disabled, button_label, slider_value, controls_style

@block18_dash_app.callback(
    Output("momentum-efficiency-tab-graph", "figure"),
    Output("momentum-efficiency-tab-graph", "style"),
    Output("shared-view-status", "children"),
    Output("momentum-efficiency-tab-graph", "animate"),
    Input("momentum-efficiency-view-tabs", "value"),
    Input("shared-update-button", "n_clicks"),
    Input("shared-display-range-dropdown", "value"),
    Input("benchmark-index-dropdown", "value"),
    Input("diagnostics-asof-slider", "value"),
    Input("diagnostics-window-range-slider", "value"),
    Input("diagnostics-spline-toggle", "value"),
    Input("diagnostics-raw-overlay-toggle", "value"),
    Input("diagnostics-spline-strength-slider", "value"),
    State("shared-window-input", "value"),
)
def _update_momentum_efficiency_dashboard(
    active_tab, _n_clicks, display_range_value, selected_benchmarks, playback_position, diagnostics_window_range, spline_toggle, raw_overlay_toggle, spline_strength, window
):
    selected_benchmarks = _block18_normalize_benchmark_selection(selected_benchmarks)
    if ctx.triggered_id == "diagnostics-window-range-slider" and active_tab == "window_diagnostics":
        patched_figure, diagnostics_window_range = _block18_patch_diagnostics_window_range(
            diagnostics_window_range
        )
        as_of_date, _ = _block18_playback_date(playback_position)
        status = (
            "Functional Horizon Profile | Profile and derivative playback"
            + " | Benchmarks: "
            + _block18_benchmark_selection_label(selected_benchmarks)
            + f" | Windows: {diagnostics_window_range[0]}-{diagnostics_window_range[1]} days"
            + f" | As of: {as_of_date:%Y-%m-%d}"
        )
        return patched_figure, no_update, status, False
    if ctx.triggered_id == "diagnostics-asof-slider" and active_tab == "window_diagnostics":
        patched_figure, as_of_date = _block18_patch_window_diagnostics(
            playback_position, selected_benchmarks, spline_toggle, spline_strength,
            raw_overlay_toggle,
        )
        status = (
            "Functional Horizon Profile | Profile and derivative playback"
            + " | Benchmarks: "
            + _block18_benchmark_selection_label(selected_benchmarks)
            + (
                f" | Cubic B-spline: {_block18_normalize_spline_strength(spline_strength):.0f}%"
                if _block18_spline_is_enabled(spline_toggle) else " | Cubic B-spline: Off"
            )
            + (
                " | Raw overlay: On"
                if _block18_spline_is_enabled(spline_toggle)
                and _block18_raw_overlay_is_enabled(raw_overlay_toggle) else ""
            )
            + f" | As of: {as_of_date:%Y-%m-%d}"
        )
        return patched_figure, no_update, status, True
    figure, status = _block18_render_dashboard_figure(
        active_tab,
        window,
        display_range_value,
        selected_benchmarks,
        playback_position=playback_position,
        diagnostics_window_range=diagnostics_window_range,
        spline_toggle=spline_toggle,
        spline_strength=spline_strength,
        raw_overlay_toggle=raw_overlay_toggle,
        cache_version=_n_clicks,
    )
    default_height = block18_tab_config.get(active_tab, {}).get("height", 1125)
    return (
        figure,
        {"height": _block18_graph_height(figure, default_height)},
        status,
        False,  # Full tab/control updates must use Plotly.react, not animation.
    )

block18_dash_app.run(
    host="127.0.0.1",
    port=_block18_available_port(),
    debug=False,
    use_reloader=False,
    jupyter_mode="inline",
    jupyter_height=3340,
)
